# RHNA & Housing Production

RHNA targets and housing production (permits/completions) for the 18
incorporated jurisdictions in San Diego County plus the County itself,
pulled from HCD, DOF, City of San Diego, and Census sources.

Target year: 2025 for RHNA/APR/DOF/permits. ACS uses the 2020-2024
5-year vintage (its most recent release).


## Setup

In [95]:
%pip install pandas numpy requests openpyxl

Note: you may need to restart the kernel to use updated packages.


In [96]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import requests
from IPython.display import display


def find_workstream_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for candidate in candidates:
        if (candidate / "notebooks").exists():
            return candidate
        my_folder = candidate / "van's work"
        if (my_folder / "notebooks").exists():
            return my_folder
    raise FileNotFoundError(
        "Could not locate workstream root (no notebooks/ folder found nearby)."
    )


ROOT = find_workstream_root()
RAW_DIR = ROOT / "data" / "raw" / "hcd"
PROCESSED_DIR = ROOT / "data" / "processed"
DOCS_DIR = ROOT / "docs"

for d in (RAW_DIR, PROCESSED_DIR, DOCS_DIR):
    d.mkdir(parents=True, exist_ok=True)

from datetime import date
DOWNLOAD_DATE = date.today().isoformat()

print("Workstream root:", ROOT)
print("Download date:", DOWNLOAD_DATE)


Workstream root: /Users/ice/Documents/GitHub/chpd-dashboard-data-validation/van's work
Download date: 2026-08-11


In [97]:
TARGET_YEAR = 2025
ACS_VINTAGE_LABEL = "2020-2024"
ACS_DATA_YEAR = 2024

SAN_DIEGO_CITIES = [
    "Carlsbad", "Chula Vista", "Coronado", "Del Mar", "El Cajon",
    "Encinitas", "Escondido", "Imperial Beach", "La Mesa", "Lemon Grove",
    "National City", "Oceanside", "Poway", "San Diego", "San Marcos",
    "Santee", "Solana Beach", "Vista",
]
COUNTY_JURISDICTION_NAME = "San Diego County"


In [98]:
COUNTY_NAME_VARIANTS = {
    "san diego county", "county of san diego", "s d county",
    "county san diego", "unincorporated", "unincorporated san diego county",
}


def normalize_jurisdiction(name: object) -> str:
    s = str(name).strip().lower()
    if s == "national city":
        return "national city"
    if s in COUNTY_NAME_VARIANTS:
        return "unincorporated san diego county"
    s = re.sub(r"^city\s+of\s+", "", s)
    s = re.sub(r"^county\s+of\s+", "county ", s)
    s = re.sub(r"\s+city$", "", s)
    s = re.sub(r"[^a-z0-9]+", " ", s).strip()
    return s


sd_jur_keys = {normalize_jurisdiction(c) for c in SAN_DIEGO_CITIES}
sd_jur_keys.add(normalize_jurisdiction(COUNTY_JURISDICTION_NAME))


def get_json(url: str, params: dict | None = None, timeout: int = 60):
    response = requests.get(
        url, params=params, timeout=timeout,
        headers={"User-Agent": "CHPD Housing Dashboard Data Validation"},
    )
    if not response.ok:
        raise RuntimeError(f"Request failed ({response.status_code}): {response.url}")
    return response.json()


def find_resource_download_url(package_json: dict, name_contains: str):
    resources = package_json["result"]["resources"]
    matches = [
        r for r in resources
        if name_contains.lower() in r.get("name", "").lower()
        and r.get("format", "").upper() == "CSV"
    ]
    if not matches:
        raise ValueError(f"No CSV resource matching '{name_contains}' found.")
    matches.sort(key=lambda r: r.get("last_modified", ""), reverse=True)
    return matches[0]["url"], matches[0]["name"]


def fill_missing_jurisdiction_years(df: pd.DataFrame, years: list) -> pd.DataFrame:
    """
    Guarantee every (jurisdiction, year) combination exists, filling 0 where
    a jurisdiction reported nothing that year - groupby silently omits
    combinations with zero rows, which reads as a missing/blank value in
    Power BI instead of a real, meaningful zero.
    """
    full_index = pd.MultiIndex.from_product([sorted(sd_jur_keys), years], names=["jur_clean", "year"])
    filled = df.set_index(["jur_clean", "year"]).reindex(full_index).reset_index()
    fill_cols = [c for c in filled.select_dtypes(include="number").columns if "share" not in c and "pct" not in c]
    filled[fill_cols] = filled[fill_cols].fillna(0)
    return filled


## APR (permits, entitlements, completions)

In [99]:
CKAN_PACKAGE_URL = (
    "https://data.ca.gov/api/3/action/package_show"
    "?id=housing-element-annual-progress-report-apr-data-by-jurisdiction-and-year"
)
package_json = get_json(CKAN_PACKAGE_URL)
table_a2_url, table_a2_name = find_resource_download_url(package_json, "Table A2")

raw_path = RAW_DIR / "apr_table_a2_raw.csv"
if not raw_path.exists():
    resp = requests.get(table_a2_url, timeout=300)
    resp.raise_for_status()
    raw_path.write_bytes(resp.content)

apr_raw = pd.read_csv(raw_path, low_memory=False)
print(apr_raw.shape)
apr_raw.head()


(920915, 69)


,JURIS_NAME,CNTY_NAME,YEAR,PRIOR_APN,APN,STREET_ADDRESS,PROJECT_NAME,JURS_TRACKING_ID,UNIT_CAT,TENURE,...,DEM_DES_UNITS_OWN_RENT,DENSITY_BONUS_TOTAL,DENSITY_BONUS_NUMBER_OTHER_INCENTIVES,DENSITY_BONUS_INCENTIVES,DENSITY_BONUS_RECEIVE_REDUCTION,NOTES,LATITUDE,LONGITUDE,STD_ADDRESS,SCORE
0,STANISLAUS COUNTY,Stanislaus,2020,NaN,081-002-032,1464 CLARK RD,NaN,BLD2002-01052,MH,Renter,...,0,0.0,0,NaN,NaN,MOBILE HOME ON PRIVATE PROPERTY (PIERS) 2002 G...,37.655693,-121.085881,"1464 Clark Rd, Modesto, California, 95358",93.96
1,STANISLAUS COUNTY,Stanislaus,2020,NaN,002-024-046,15363 ORANGE BLOSSOM RD,NaN,BLD2020-0101,MH,Renter,...,0,0.0,0,NaN,NaN,TEMPORARY 1624 SQ FT MANUFACTURED DWELLING - 2...,37.815710,-120.715241,"15363 Orange Blossom Rd, Oakdale, California, ...",89.68
2,STANISLAUS COUNTY,Stanislaus,2020,NaN,048-006-007,2333 FIG AVE,NaN,BLD2020-1248,MH,Renter,...,0,0.0,0,NaN,NaN,(( ACA )) MANUFACTURED HOME 2020 CHAMPION MODE...,37.477489,-121.080866,"2333 Fig Ave, Patterson, California, 95363",87.69
3,STANISLAUS COUNTY,Stanislaus,2020,NaN,001-009-015,6990 State Route 4,NaN,BLD2018-1774,MH,Owner,...,0,0.0,0,NaN,NaN,MANUFACTURED HOME / 2018 CMH MODEL FAIRPOINT 2...,37.490499,-120.848087,"4th St, Turlock, California, 95380",80.09
4,STANISLAUS COUNTY,Stanislaus,2020,NaN,062-027-003,5314 LANGWORTH,NaN,BLD2019-1749,MH,Renter,...,0,0.0,0,NaN,NaN,TEMPORARY 1440 SQ FT MANUFACTED DWELLING 1973 ...,37.717510,-120.894040,"5314 Langworth Rd, Oakdale, California, 95361",87.17


In [100]:
print(apr_raw.columns.tolist())

['JURIS_NAME', 'CNTY_NAME', 'YEAR', 'PRIOR_APN', 'APN', 'STREET_ADDRESS', 'PROJECT_NAME', 'JURS_TRACKING_ID', 'UNIT_CAT', 'TENURE', 'ACUTELY_LOW_INCOME_DR', 'ACUTELY_LOW_INCOME_NDR', 'EXTREMELY_LOW_INCOME_DR', 'EXTREMELY_LOW_INCOME_NDR', 'VLOW_INCOME_DR', 'VLOW_INCOME_NDR', 'LOW_INCOME_DR', 'LOW_INCOME_NDR', 'MOD_INCOME_DR', 'MOD_INCOME_NDR', 'ABOVE_MOD_INCOME', 'ENT_APPROVE_DT1', 'NO_ENTITLEMENTS', 'BP_ACUTELY_LOW_INCOME_DR', 'BP_ACUTELY_LOW_INCOME_NDR', 'BP_EXTREMELY_LOW_INCOME_DR', 'BP_EXTREMELY_LOW_INCOME_NDR', 'BP_VLOW_INCOME_DR', 'BP_VLOW_INCOME_NDR', 'BP_LOW_INCOME_DR', 'BP_LOW_INCOME_NDR', 'BP_MOD_INCOME_DR', 'BP_MOD_INCOME_NDR', 'BP_ABOVE_MOD_INCOME', 'BP_ISSUE_DT1', 'NO_BUILDING_PERMITS', 'CO_ACUTELY_LOW_INCOME_DR', 'CO_ACUTELY_LOW_INCOME_NDR', 'CO_EXTREMELY_LOW_INCOME_DR', 'CO_EXTREMELY_LOW_INCOME_NDR', 'CO_VLOW_INCOME_DR', 'CO_VLOW_INCOME_NDR', 'CO_LOW_INCOME_DR', 'CO_LOW_INCOME_NDR', 'CO_MOD_INCOME_DR', 'CO_MOD_INCOME_NDR', 'CO_ABOVE_MOD_INCOME', 'CO_ISSUE_DT1', 'NO_OTHER_

**Development stages tracked here, kept strictly separate:**

| Stage | Source | Metric |
|---|---|---|
| Application submitted | APR Table A | `application_units_total` |
| Entitlement | APR Table A2 (unprefixed `*_INCOME_*` columns; cross-checked against HCD's own auto-populated `NO_ENTITLEMENTS` total) | `ent_units_total` |
| Building permit | APR Table A2 (`BP_*_INCOME` columns; cross-checked against `NO_BUILDING_PERMITS`) | `bp_units_total` |
| Completion (certificate of occupancy) | APR Table A2 (`CO_*_INCOME` columns; cross-checked against `NO_OTHER_FORMS_OF_READINESS`) | `co_units_total` |

**Units under construction** is the one stage from the dashboard reorg
guidance that is **not tracked anywhere in HCD's APR data** - no table
captures this milestone.

Never summed into one another - a project can appear as an
application one year, entitled the next, permitted the year after,
so combining stages double-counts the same units.

In [101]:
JURISDICTION_COL = "JURIS_NAME"
YEAR_COL = "YEAR"

apr_raw["jur_clean"] = apr_raw[JURISDICTION_COL].map(normalize_jurisdiction)
apr_raw[YEAR_COL] = pd.to_numeric(apr_raw[YEAR_COL], errors="coerce")

sd_apr = apr_raw[apr_raw["jur_clean"].isin(sd_jur_keys)].copy()
sd_apr_target_year = sd_apr[sd_apr[YEAR_COL] == TARGET_YEAR].copy()

print("SD rows, all years:", len(sd_apr))
print(f"SD rows, {TARGET_YEAR}:", len(sd_apr_target_year))
missing = sd_jur_keys - set(sd_apr_target_year["jur_clean"].unique())
if missing:
    print(f"No {TARGET_YEAR} row yet:", sorted(missing))


SD rows, all years: 52514
SD rows, 2025: 7460


In [102]:
# Table A2 tracks THREE separate stages, not two: unprefixed *_INCOME_*
# columns are entitlement-stage units (paired with ENT_APPROVE_DT1), BP_*
# are building permits, CO_* are completions. Kept as three fully separate
# metrics - never summed into one another.
#
# Same 4-tier grouping used for RHNA (very_low/low/moderate/above_moderate)
# - rolls Acutely Low + Extremely Low + Very Low columns into "very_low",
# matching HCD's own convention that those sub-tiers count toward
# very-low-income reporting.
TIER_SUFFIX_GROUPS = {
    "very_low": [
        "ACUTELY_LOW_INCOME_DR", "ACUTELY_LOW_INCOME_NDR",
        "EXTREMELY_LOW_INCOME_DR", "EXTREMELY_LOW_INCOME_NDR",
        "VLOW_INCOME_DR", "VLOW_INCOME_NDR",
    ],
    "low": ["LOW_INCOME_DR", "LOW_INCOME_NDR"],
    "moderate": ["MOD_INCOME_DR", "MOD_INCOME_NDR"],
    "above_moderate": ["ABOVE_MOD_INCOME"],
}
ALL_TIER_SUFFIXES = [s for suffixes in TIER_SUFFIX_GROUPS.values() for s in suffixes]

# Built explicitly from the known tier structure, not a loose "contains
# INCOME" text search - Table A2 also has EXTR_LOW_INCOME_UNITS, an
# unrelated field that a loose search would incorrectly sweep in.
ENT_INCOME_COLS = [s for s in ALL_TIER_SUFFIXES if s in sd_apr_target_year.columns]
BP_INCOME_COLS = [f"BP_{s}" for s in ALL_TIER_SUFFIXES if f"BP_{s}" in sd_apr_target_year.columns]
CO_INCOME_COLS = [f"CO_{s}" for s in ALL_TIER_SUFFIXES if f"CO_{s}" in sd_apr_target_year.columns]

sd_apr_target_year["ent_units_row"] = sd_apr_target_year[ENT_INCOME_COLS].sum(axis=1, numeric_only=True)
sd_apr_target_year["bp_units_row"] = sd_apr_target_year[BP_INCOME_COLS].sum(axis=1, numeric_only=True)
sd_apr_target_year["co_units_row"] = sd_apr_target_year[CO_INCOME_COLS].sum(axis=1, numeric_only=True)

above_mod_ent = [c for c in ENT_INCOME_COLS if "ABOVE" in c.upper()]
above_mod_bp = [c for c in BP_INCOME_COLS if "ABOVE" in c.upper()]
above_mod_co = [c for c in CO_INCOME_COLS if "ABOVE" in c.upper()]

sd_apr_target_year["ent_affordable_row"] = (
    sd_apr_target_year["ent_units_row"] - sd_apr_target_year[above_mod_ent].sum(axis=1, numeric_only=True)
)
sd_apr_target_year["bp_affordable_row"] = (
    sd_apr_target_year["bp_units_row"] - sd_apr_target_year[above_mod_bp].sum(axis=1, numeric_only=True)
)
sd_apr_target_year["co_affordable_row"] = (
    sd_apr_target_year["co_units_row"] - sd_apr_target_year[above_mod_co].sum(axis=1, numeric_only=True)
)

# Per-tier entitlement, permit, and completion columns.
tier_agg_kwargs = {}
for tier, suffixes in TIER_SUFFIX_GROUPS.items():
    ent_cols = [s for s in suffixes if s in sd_apr_target_year.columns]
    bp_cols = [f"BP_{s}" for s in suffixes if f"BP_{s}" in sd_apr_target_year.columns]
    co_cols = [f"CO_{s}" for s in suffixes if f"CO_{s}" in sd_apr_target_year.columns]
    sd_apr_target_year[f"ent_{tier}_row"] = sd_apr_target_year[ent_cols].sum(axis=1, numeric_only=True)
    sd_apr_target_year[f"bp_{tier}_row"] = sd_apr_target_year[bp_cols].sum(axis=1, numeric_only=True)
    sd_apr_target_year[f"co_{tier}_row"] = sd_apr_target_year[co_cols].sum(axis=1, numeric_only=True)
    tier_agg_kwargs[f"ent_{tier}_total"] = (f"ent_{tier}_row", "sum")
    tier_agg_kwargs[f"bp_{tier}_total"] = (f"bp_{tier}_row", "sum")
    tier_agg_kwargs[f"co_{tier}_total"] = (f"co_{tier}_row", "sum")

# The raw file is row-level (one row per project/address) - group up to
# jurisdiction-year before this becomes a usable production metric.
production_by_year = (
    sd_apr_target_year
    .groupby("jur_clean", as_index=False)
    .agg(
        year=(YEAR_COL, "first"),
        ent_units_total=("ent_units_row", "sum"),
        bp_units_total=("bp_units_row", "sum"),
        co_units_total=("co_units_row", "sum"),
        ent_affordable_total=("ent_affordable_row", "sum"),
        bp_affordable_total=("bp_affordable_row", "sum"),
        co_affordable_total=("co_affordable_row", "sum"),
        project_rows=("jur_clean", "size"),
        **tier_agg_kwargs,
    )
)
production_by_year["ent_affordable_share"] = production_by_year["ent_affordable_total"] / production_by_year["ent_units_total"]
production_by_year["bp_affordable_share"] = production_by_year["bp_affordable_total"] / production_by_year["bp_units_total"]
production_by_year["co_affordable_share"] = production_by_year["co_affordable_total"] / production_by_year["co_units_total"]

# Sanity check: per-tier ent/bp/co columns should sum back to the overall total.
tier_ent_sum = production_by_year[[f"ent_{t}_total" for t in TIER_SUFFIX_GROUPS]].sum(axis=1)
tier_bp_sum = production_by_year[[f"bp_{t}_total" for t in TIER_SUFFIX_GROUPS]].sum(axis=1)
tier_co_sum = production_by_year[[f"co_{t}_total" for t in TIER_SUFFIX_GROUPS]].sum(axis=1)
assert (tier_ent_sum == production_by_year["ent_units_total"]).all(), "ENT tier columns do not sum to ent_units_total"
assert (tier_bp_sum == production_by_year["bp_units_total"]).all(), "BP tier columns do not sum to bp_units_total"
assert (tier_co_sum == production_by_year["co_units_total"]).all(), "CO tier columns do not sum to co_units_total"

production_by_year


,jur_clean,year,ent_units_total,bp_units_total,co_units_total,ent_affordable_total,bp_affordable_total,co_affordable_total,project_rows,ent_very_low_total,...,co_low_total,ent_moderate_total,bp_moderate_total,co_moderate_total,ent_above_moderate_total,bp_above_moderate_total,co_above_moderate_total,ent_affordable_share,bp_affordable_share,co_affordable_share
0,carlsbad,2025,202,343,719,13,36,141,428,11,...,84,2,1,12,189,307,578,0.064356,0.104956,0.196106
1,chula vista,2025,1323,723,1428,0,224,201,663,0,...,1,0,222,200,1323,499,1227,0.000000,0.309820,0.140756
2,coronado,2025,27,24,36,0,0,0,54,0,...,0,0,0,0,27,24,36,0.000000,0.000000,0.000000
3,del mar,2025,15,14,17,12,11,10,43,0,...,0,12,11,10,3,3,7,0.800000,0.785714,0.588235
4,el cajon,2025,99,210,235,40,83,123,164,0,...,99,36,45,24,59,127,112,0.404040,0.395238,0.523404
5,encinitas,2025,277,187,228,40,33,43,461,0,...,3,9,14,15,237,154,185,0.144404,0.176471,0.188596
6,escondido,2025,366,514,604,234,246,66,396,119,...,30,6,11,4,132,268,538,0.639344,0.478599,0.109272
7,imperial beach,2025,45,71,13,0,0,0,86,0,...,0,0,0,0,45,71,13,0.000000,0.000000,0.000000
8,la mesa,2025,78,254,249,11,99,215,223,0,...,93,11,69,62,67,155,34,0.141026,0.389764,0.863454
9,lemon grove,2025,1,32,61,0,6,8,77,0,...,8,0,0,0,1,26,53,0.000000,0.187500,0.131148


### Cross-check against HCD's auto-populated totals

`NO_ENTITLEMENTS` / `NO_BUILDING_PERMITS` / `NO_OTHER_FORMS_OF_READINESS`
mean "Number Of," not a flag - HCD auto-populates these from the same
per-tier income columns summed below. Used here as an independent check
on ent/bp/co totals.

In [103]:
validation_cols = {
    "ent_units_total": "NO_ENTITLEMENTS",
    "bp_units_total": "NO_BUILDING_PERMITS",
    "co_units_total": "NO_OTHER_FORMS_OF_READINESS",
}

hcd_totals = sd_apr_target_year.groupby("jur_clean", as_index=False)[list(validation_cols.values())].sum()
stage_check = production_by_year.merge(hcd_totals, on="jur_clean")

for our_col, hcd_col in validation_cols.items():
    stage_check[f"{our_col}_diff"] = stage_check[our_col] - stage_check[hcd_col]

diff_cols = [f"{c}_diff" for c in validation_cols]
print("Max absolute difference per stage:")
print(stage_check[diff_cols].abs().max())
stage_check[["jur_clean"] + list(validation_cols.keys()) + list(validation_cols.values()) + diff_cols]


Max absolute difference per stage:
ent_units_total_diff    0
bp_units_total_diff     0
co_units_total_diff     0
dtype: int64


,jur_clean,ent_units_total,bp_units_total,co_units_total,NO_ENTITLEMENTS,NO_BUILDING_PERMITS,NO_OTHER_FORMS_OF_READINESS,ent_units_total_diff,bp_units_total_diff,co_units_total_diff
0,carlsbad,202,343,719,202,343,719,0,0,0
1,chula vista,1323,723,1428,1323,723,1428,0,0,0
2,coronado,27,24,36,27,24,36,0,0,0
3,del mar,15,14,17,15,14,17,0,0,0
4,el cajon,99,210,235,99,210,235,0,0,0
5,encinitas,277,187,228,277,187,228,0,0,0
6,escondido,366,514,604,366,514,604,0,0,0
7,imperial beach,45,71,13,45,71,13,0,0,0
8,la mesa,78,254,249,78,254,249,0,0,0
9,lemon grove,1,32,61,1,32,61,0,0,0


### Historical range check (2018-2025)

APR data collection began in 2018. Confirms every year is actually
present for San Diego County, not just assumed.

In [104]:
APR_START_YEAR = 2018
APR_YEARS = list(range(APR_START_YEAR, TARGET_YEAR + 1))

ENT_INCOME_COLS_ALL = [s for s in ALL_TIER_SUFFIXES if s in sd_apr.columns]
BP_INCOME_COLS_ALL = [f"BP_{s}" for s in ALL_TIER_SUFFIXES if f"BP_{s}" in sd_apr.columns]
CO_INCOME_COLS_ALL = [f"CO_{s}" for s in ALL_TIER_SUFFIXES if f"CO_{s}" in sd_apr.columns]

sd_apr["ent_units_row"] = sd_apr[ENT_INCOME_COLS_ALL].sum(axis=1, numeric_only=True)
sd_apr["bp_units_row"] = sd_apr[BP_INCOME_COLS_ALL].sum(axis=1, numeric_only=True)
sd_apr["co_units_row"] = sd_apr[CO_INCOME_COLS_ALL].sum(axis=1, numeric_only=True)

above_mod_ent_all = [c for c in ENT_INCOME_COLS_ALL if "ABOVE" in c.upper()]
above_mod_bp_all = [c for c in BP_INCOME_COLS_ALL if "ABOVE" in c.upper()]
above_mod_co_all = [c for c in CO_INCOME_COLS_ALL if "ABOVE" in c.upper()]

sd_apr["ent_affordable_row"] = sd_apr["ent_units_row"] - sd_apr[above_mod_ent_all].sum(axis=1, numeric_only=True)
sd_apr["bp_affordable_row"] = sd_apr["bp_units_row"] - sd_apr[above_mod_bp_all].sum(axis=1, numeric_only=True)
sd_apr["co_affordable_row"] = sd_apr["co_units_row"] - sd_apr[above_mod_co_all].sum(axis=1, numeric_only=True)

tier_agg_kwargs_hist = {}
for tier, suffixes in TIER_SUFFIX_GROUPS.items():
    ent_cols = [s for s in suffixes if s in sd_apr.columns]
    bp_cols = [f"BP_{s}" for s in suffixes if f"BP_{s}" in sd_apr.columns]
    co_cols = [f"CO_{s}" for s in suffixes if f"CO_{s}" in sd_apr.columns]
    sd_apr[f"ent_{tier}_row"] = sd_apr[ent_cols].sum(axis=1, numeric_only=True)
    sd_apr[f"bp_{tier}_row"] = sd_apr[bp_cols].sum(axis=1, numeric_only=True)
    sd_apr[f"co_{tier}_row"] = sd_apr[co_cols].sum(axis=1, numeric_only=True)
    tier_agg_kwargs_hist[f"ent_{tier}_total"] = (f"ent_{tier}_row", "sum")
    tier_agg_kwargs_hist[f"bp_{tier}_total"] = (f"bp_{tier}_row", "sum")
    tier_agg_kwargs_hist[f"co_{tier}_total"] = (f"co_{tier}_row", "sum")

production_history = (
    sd_apr[sd_apr[YEAR_COL].isin(APR_YEARS)]
    .groupby(["jur_clean", YEAR_COL], as_index=False)
    .agg(
        ent_units_total=("ent_units_row", "sum"),
        bp_units_total=("bp_units_row", "sum"),
        co_units_total=("co_units_row", "sum"),
        ent_affordable_total=("ent_affordable_row", "sum"),
        bp_affordable_total=("bp_affordable_row", "sum"),
        co_affordable_total=("co_affordable_row", "sum"),
        project_rows=("jur_clean", "size"),
        **tier_agg_kwargs_hist,
    )
    .rename(columns={YEAR_COL: "year"})
)
production_history["ent_affordable_share"] = production_history["ent_affordable_total"] / production_history["ent_units_total"]
production_history["bp_affordable_share"] = production_history["bp_affordable_total"] / production_history["bp_units_total"]
production_history["co_affordable_share"] = production_history["co_affordable_total"] / production_history["co_units_total"]

print(production_history.shape)
production_history.head()


(152, 24)


,jur_clean,year,ent_units_total,bp_units_total,co_units_total,ent_affordable_total,bp_affordable_total,co_affordable_total,project_rows,ent_very_low_total,...,co_low_total,ent_moderate_total,bp_moderate_total,co_moderate_total,ent_above_moderate_total,bp_above_moderate_total,co_above_moderate_total,ent_affordable_share,bp_affordable_share,co_affordable_share
0,carlsbad,2018,63,243,218,15,33,25,150,0,...,4,12,28,21,48,210,193,0.238095,0.135802,0.114679
1,carlsbad,2019,46,322,548,5,110,189,121,0,...,111,5,59,78,41,212,359,0.108696,0.341615,0.344891
2,carlsbad,2020,756,377,551,147,117,49,141,7,...,23,13,39,24,609,260,502,0.194444,0.310345,0.088929
3,carlsbad,2021,362,150,409,81,74,121,173,0,...,50,18,65,70,281,76,288,0.223757,0.493333,0.295844
4,carlsbad,2022,52,125,220,19,82,115,286,0,...,2,19,82,65,33,43,105,0.365385,0.656000,0.522727


In [105]:
years_present = sorted(production_history["year"].dropna().unique())
years_missing = sorted(set(APR_YEARS) - set(years_present))
print("Years present:", years_present)
if years_missing:
    print("Years missing entirely from the pull:", years_missing)

coverage = (
    production_history
    .groupby("year")["jur_clean"]
    .agg(jurisdictions="nunique", rows="count")
    .reindex(APR_YEARS)
)
coverage["missing_jurisdictions"] = coverage["jurisdictions"].apply(
    lambda n: 19 - n if pd.notna(n) else 19
)
coverage


Years present: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


,jurisdictions,rows,missing_jurisdictions
year,,,
2018,19,19,0
2019,19,19,0
2020,19,19,0
2021,19,19,0
2022,19,19,0
2023,19,19,0
2024,19,19,0
2025,19,19,0


In [106]:
# Reindex AFTER the coverage check above, so the check reflects real
# reporting gaps - this fill is only for downstream use (Power BI export).
production_history = fill_missing_jurisdiction_years(production_history, APR_YEARS)
print(production_history.shape)


(152, 24)


In [107]:
history_output_path = PROCESSED_DIR / f"apr_production_{APR_START_YEAR}_{TARGET_YEAR}_by_jurisdiction_year.csv"
production_history.to_csv(history_output_path, index=False)
print("Saved:", history_output_path)


Saved: /Users/ice/Documents/GitHub/chpd-dashboard-data-validation/van's work/data/processed/apr_production_2018_2025_by_jurisdiction_year.csv


## APR Table A (applications)

Applications live in a separate table from entitlements/permits/completions
(Table A vs. Table A2). Loaded the same way as Table A2 - CKAN action API,
resolved by exact resource name.

In [108]:
table_a_package_json = get_json(CKAN_PACKAGE_URL)

table_a_matches = [
    r for r in table_a_package_json["result"]["resources"]
    if r.get("name", "").strip().lower() == "apr table a"
    and r.get("format", "").upper() == "CSV"
]
if not table_a_matches:
    raise ValueError("No exact 'APR Table A' CSV resource found in the package.")
table_a_url = table_a_matches[0]["url"]
table_a_name = table_a_matches[0]["name"]
print("Using resource:", table_a_name)
print("Download URL:", table_a_url)

table_a_raw_path = RAW_DIR / "apr_table_a_raw.csv"
if not table_a_raw_path.exists():
    resp = requests.get(table_a_url, timeout=300)
    resp.raise_for_status()
    table_a_raw_path.write_bytes(resp.content)

apr_table_a_raw = pd.read_csv(table_a_raw_path, low_memory=False)
print(apr_table_a_raw.shape)
apr_table_a_raw.head()


Using resource: APR Table A
Download URL: https://data.ca.gov/dataset/81b0841f-2802-403e-b48e-2ef4b751f77c/resource/c78b769d-cc02-4050-91ef-79ded665b5a8/download/tablea.csv
(357870, 36)


,JURIS_NAME,CNTY_NAME,YEAR,PRIOR_APN,APN,STREET_ADDRESS,PROJECT_NAME,JURS_TRACKING_ID,UNIT_CAT,TENURE,...,HISTORIC_SITES,DENSITY_BONUS_RECEIVED,DENSITY_BONUS_APPROVED,APPLICATION_STATUS,PROJECT_TYPE,NOTES,LATITUDE,LONGITUDE,STD_ADDRESS,SCORE
0,PETALUMA,Sonoma,2021,NaN,136-690-007,500 HOPPER ST\nPETALUM...,Riverscape Townhomes,NaN,2 to 4,Owner,...,NaN,No,No,Pending,NaN,In entitlement review,38.234548,-122.627379,"500 Hopper St, Petaluma, California, 94952",98.52
1,LONG BEACH,Los Angeles,2021,NaN,7243021019,10 Virgil Walk,2102-11,2102-11,ADU,Renter,...,NaN,No,NaN,Approved,NaN,NaN,33.751905,-118.122695,"10 Virgil Walk, Long Beach, California, 90803",100.00
2,LONG BEACH,Los Angeles,2021,NaN,7272003017,1127 Magnolia Avenue,2102-08,2102-08,ADU,Renter,...,NaN,No,NaN,Approved,NaN,NaN,33.780730,-118.198063,"1127 Magnolia Ave, Long Beach, California, 90813",100.00
3,LONG BEACH,Los Angeles,2021,NaN,7265014029,1412 E. 1st St.,2102-01,2102-01,ADU,Renter,...,NaN,No,NaN,Approved,NaN,NaN,33.766344,-118.174140,"1412 E 1st St, Long Beach, California, 90802",100.00
4,LONG BEACH,Los Angeles,2021,NaN,7256022032,147 Park Ave.,2105-08,2105-08,ADU,Renter,...,NaN,No,NaN,Approved,NaN,NaN,33.760005,-118.139042,"147 Park Ave, Long Beach, California, 90803",100.00


In [109]:
print(apr_table_a_raw.columns.tolist())

['JURIS_NAME', 'CNTY_NAME', 'YEAR', 'PRIOR_APN', 'APN', 'STREET_ADDRESS', 'PROJECT_NAME', 'JURS_TRACKING_ID', 'UNIT_CAT', 'TENURE', 'APP_SUBMIT_DT', 'ACUTELY_LOW_INCOME_DR', 'ACUTELY_LOW_INCOME_NDR', 'EXTREMELY_LOW_INCOME_DR', 'EXTREMELY_LOW_INCOME_NDR', 'VLOW_INCOME_DR', 'VLOW_INCOME_NDR', 'LOW_INCOME_DR', 'LOW_INCOME_NDR', 'MOD_INCOME_DR', 'MOD_INCOME_NDR', 'ABOVE_MOD_INCOME', 'TOT_PROPOSED_UNITS', 'TOT_APPROVED_UNITS', 'TOT_DISAPPROVED_UNITS', 'APP_SUBMITTED_SB35', 'HISTORIC_SITES', 'DENSITY_BONUS_RECEIVED', 'DENSITY_BONUS_APPROVED', 'APPLICATION_STATUS', 'PROJECT_TYPE', 'NOTES', 'LATITUDE', 'LONGITUDE', 'STD_ADDRESS', 'SCORE']


In [110]:
TABLE_A_JUR_COL = "JURIS_NAME"
TABLE_A_YEAR_COL = "YEAR"

apr_table_a_raw["jur_clean"] = apr_table_a_raw[TABLE_A_JUR_COL].map(normalize_jurisdiction)
apr_table_a_raw[TABLE_A_YEAR_COL] = pd.to_numeric(apr_table_a_raw[TABLE_A_YEAR_COL], errors="coerce")

sd_apr_table_a = apr_table_a_raw[apr_table_a_raw["jur_clean"].isin(sd_jur_keys)].copy()
sd_apr_table_a_target_year = sd_apr_table_a[sd_apr_table_a[TABLE_A_YEAR_COL] == TARGET_YEAR].copy()

print("SD rows, all years:", len(sd_apr_table_a))
print(f"SD rows, {TARGET_YEAR}:", len(sd_apr_table_a_target_year))
missing = sd_jur_keys - set(sd_apr_table_a_target_year["jur_clean"].unique())
if missing:
    print(f"No {TARGET_YEAR} row yet:", sorted(missing))


SD rows, all years: 23118
SD rows, 2025: 4874


In [111]:
APPLICATION_INCOME_COLS = [
    "ACUTELY_LOW_INCOME_DR", "ACUTELY_LOW_INCOME_NDR",
    "EXTREMELY_LOW_INCOME_DR", "EXTREMELY_LOW_INCOME_NDR",
    "VLOW_INCOME_DR", "VLOW_INCOME_NDR",
    "LOW_INCOME_DR", "LOW_INCOME_NDR",
    "MOD_INCOME_DR", "MOD_INCOME_NDR",
    "ABOVE_MOD_INCOME",
]
above_mod_application = [c for c in APPLICATION_INCOME_COLS if "ABOVE" in c.upper()]

sd_apr_table_a_target_year["application_units_row"] = (
    sd_apr_table_a_target_year[APPLICATION_INCOME_COLS].sum(axis=1, numeric_only=True)
)
sd_apr_table_a_target_year["application_affordable_row"] = (
    sd_apr_table_a_target_year["application_units_row"]
    - sd_apr_table_a_target_year[above_mod_application].sum(axis=1, numeric_only=True)
)

application_tier_agg_kwargs = {}
for tier, suffixes in TIER_SUFFIX_GROUPS.items():
    app_cols = [s for s in suffixes if s in sd_apr_table_a_target_year.columns]
    sd_apr_table_a_target_year[f"application_{tier}_row"] = (
        sd_apr_table_a_target_year[app_cols].sum(axis=1, numeric_only=True)
    )
    application_tier_agg_kwargs[f"application_{tier}_total"] = (f"application_{tier}_row", "sum")

applications_by_jurisdiction = (
    sd_apr_table_a_target_year
    .groupby("jur_clean", as_index=False)
    .agg(
        application_units_total=("application_units_row", "sum"),
        application_affordable_total=("application_affordable_row", "sum"),
        application_rows=("jur_clean", "size"),
        **application_tier_agg_kwargs,
    )
)
applications_by_jurisdiction["application_affordable_share"] = (
    applications_by_jurisdiction["application_affordable_total"] / applications_by_jurisdiction["application_units_total"]
)

print(applications_by_jurisdiction.shape)
applications_by_jurisdiction


(19, 9)


,jur_clean,application_units_total,application_affordable_total,application_rows,application_very_low_total,application_low_total,application_moderate_total,application_above_moderate_total,application_affordable_share
0,carlsbad,813,114,103,9,104,1,699,0.140221
1,chula vista,2057,560,428,131,99,330,1497,0.272241
2,coronado,45,0,36,0,0,0,45,0.000000
3,del mar,18,15,18,0,0,15,3,0.833333
4,el cajon,241,200,148,0,66,134,41,0.829876
5,encinitas,316,19,137,2,8,9,297,0.060127
6,escondido,554,232,93,118,108,6,322,0.418773
7,imperial beach,197,55,88,28,0,27,142,0.279188
8,la mesa,453,129,148,5,19,105,324,0.284768
9,lemon grove,90,9,60,0,9,0,81,0.100000


In [112]:
proposed_check = sd_apr_table_a_target_year.groupby("jur_clean", as_index=False).agg(
    tot_proposed_units=("TOT_PROPOSED_UNITS", "sum")
)
check = applications_by_jurisdiction.merge(proposed_check, on="jur_clean")
check["diff"] = check["application_units_total"] - check["tot_proposed_units"]
check[["jur_clean", "application_units_total", "tot_proposed_units", "diff"]]


,jur_clean,application_units_total,tot_proposed_units,diff
0,carlsbad,813,813,0
1,chula vista,2057,2047,10
2,coronado,45,45,0
3,del mar,18,18,0
4,el cajon,241,241,0
5,encinitas,316,315,1
6,escondido,554,466,88
7,imperial beach,197,172,25
8,la mesa,453,453,0
9,lemon grove,90,90,0


### Historical applications (2018-2025)

Same treatment as APR Table A2 - `sd_apr_table_a` already has all years
for San Diego County; aggregate the full range rather than just
`TARGET_YEAR`.

In [113]:
sd_apr_table_a["application_units_row"] = (
    sd_apr_table_a[APPLICATION_INCOME_COLS].sum(axis=1, numeric_only=True)
)
sd_apr_table_a["application_affordable_row"] = (
    sd_apr_table_a["application_units_row"] - sd_apr_table_a[above_mod_application].sum(axis=1, numeric_only=True)
)

application_tier_agg_kwargs_hist = {}
for tier, suffixes in TIER_SUFFIX_GROUPS.items():
    app_cols = [s for s in suffixes if s in sd_apr_table_a.columns]
    sd_apr_table_a[f"application_{tier}_row"] = sd_apr_table_a[app_cols].sum(axis=1, numeric_only=True)
    application_tier_agg_kwargs_hist[f"application_{tier}_total"] = (f"application_{tier}_row", "sum")

applications_history = (
    sd_apr_table_a[sd_apr_table_a[TABLE_A_YEAR_COL].isin(APR_YEARS)]
    .groupby(["jur_clean", TABLE_A_YEAR_COL], as_index=False)
    .agg(
        application_units_total=("application_units_row", "sum"),
        application_affordable_total=("application_affordable_row", "sum"),
        application_rows=("jur_clean", "size"),
        **application_tier_agg_kwargs_hist,
    )
    .rename(columns={TABLE_A_YEAR_COL: "year"})
)
applications_history["application_affordable_share"] = (
    applications_history["application_affordable_total"] / applications_history["application_units_total"]
)

years_present_app = sorted(applications_history["year"].dropna().unique())
years_missing_app = sorted(set(APR_YEARS) - set(years_present_app))
print("Years present (applications):", years_present_app)
if years_missing_app:
    print("Years missing entirely:", years_missing_app)

coverage_app = (
    applications_history
    .groupby("year")["jur_clean"]
    .agg(jurisdictions="nunique", rows="count")
    .reindex(APR_YEARS)
)
coverage_app["missing_jurisdictions"] = coverage_app["jurisdictions"].apply(
    lambda n: 19 - n if pd.notna(n) else 19
)
print(applications_history.shape)
coverage_app


Years present (applications): [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
(149, 10)


,jurisdictions,rows,missing_jurisdictions
year,,,
2018,18,18,1
2019,18,18,1
2020,18,18,1
2021,19,19,0
2022,19,19,0
2023,19,19,0
2024,19,19,0
2025,19,19,0


In [114]:
for yr in [2018, 2019, 2020]:
    present = set(applications_history[applications_history["year"] == yr]["jur_clean"])
    missing = sd_jur_keys - present
    if missing:
        print(f"{yr}: missing {sorted(missing)}")


2018: missing ['lemon grove']
2019: missing ['imperial beach']
2020: missing ['san marcos']


In [115]:
applications_history = fill_missing_jurisdiction_years(applications_history, APR_YEARS)
print(applications_history.shape)


(152, 10)


## RHNA 6th Cycle targets

In [116]:
RHNA_PACKAGE_URL = "https://data.ca.gov/api/3/action/package_show?id=rhna-progress-report"
rhna_package_json = get_json(RHNA_PACKAGE_URL)
rhna6_url, rhna6_name = find_resource_download_url(rhna_package_json, "6th Cycle RHNA Progress Report")

rhna_raw_path = RAW_DIR / "rhna6_progress_raw.csv"
if not rhna_raw_path.exists():
    resp = requests.get(rhna6_url, timeout=120)
    resp.raise_for_status()
    rhna_raw_path.write_bytes(resp.content)

rhna_raw = pd.read_csv(rhna_raw_path, low_memory=False)
print(rhna_raw.shape)
rhna_raw.head()


(539, 15)


,Jurisdiction,Planning Period,6th Cycle Started,VLI UNITS,RHNA VLI,VLI %,LI UNITS,RHNA LI,LI %,MOD UNITS,RHNA MOD,MOD %,ABOVE MOD UNITS,RHNA ABOVE MOD,ABOVE MOD %
0,AMERICAN CANYON,01/31/2023 - 01/31/2031,True,11,169,0.07,5,109,0.05,2,95,0.02,487,249,1.96
1,AGOURA HILLS,10/15/2021 - 10/15/2029,True,44,127,0.35,10,72,0.14,6,55,0.11,277,64,4.33
2,ALPINE COUNTY,08/31/2019 - 06/30/2024,True,0,1,0.00,0,1,0.00,2,0,0.00,33,0,0.00
3,ALAMEDA,01/31/2023 - 01/31/2031,True,155,1421,0.11,47,818,0.06,55,868,0.06,192,2246,0.09
4,AMADOR,09/15/2021 - 09/15/2029,True,0,1,0.00,0,1,0.00,0,1,0.00,3,2,1.50


In [117]:
print(rhna_raw.columns.tolist())

['Jurisdiction', 'Planning Period', '6th Cycle Started', 'VLI UNITS', 'RHNA VLI', 'VLI %', 'LI UNITS', 'RHNA LI', 'LI %', 'MOD UNITS', 'RHNA MOD', 'MOD %', 'ABOVE MOD UNITS', 'RHNA ABOVE MOD', 'ABOVE MOD %']


**RHNA progress basis:** `rhna_reported_<tier>` / `rhna_pct_achieved_<tier>`
/ `rhna_remaining_<tier>` are based on **building permits issued**, per
HCD's own RHNA-credit methodology - not completions, not entitlements.
Distinct from `co_units_total` (completions); don't conflate the two.

In [118]:
RHNA_JUR_COL = "Jurisdiction"

rhna_raw["jur_clean"] = rhna_raw[RHNA_JUR_COL].map(normalize_jurisdiction)
sd_rhna6 = rhna_raw[rhna_raw["jur_clean"].isin(sd_jur_keys)].copy()

print("SD rows:", len(sd_rhna6))
missing = sd_jur_keys - set(sd_rhna6["jur_clean"].unique())
if missing:
    print("Not found:", sorted(missing))
sd_rhna6.head()


SD rows: 19


,Jurisdiction,Planning Period,6th Cycle Started,VLI UNITS,RHNA VLI,VLI %,LI UNITS,RHNA LI,LI %,MOD UNITS,RHNA MOD,MOD %,ABOVE MOD UNITS,RHNA ABOVE MOD,ABOVE MOD %,jur_clean
76,CARLSBAD,04/30/2021 - 04/30/2029,True,65,1311,0.05,198,784,0.25,293,749,0.39,991,1029,0.96,carlsbad
85,CORONADO,04/30/2021 - 04/30/2029,True,0,312,0.00,0,169,0.00,0,159,0.00,204,272,0.75,coronado
92,DEL MAR,04/30/2021 - 04/30/2029,True,0,37,0.00,0,64,0.00,66,31,2.13,45,31,1.45,del mar
99,EL CAJON,04/30/2021 - 04/30/2029,True,0,481,0.00,281,414,0.68,154,518,0.30,350,1867,0.19,el cajon
151,CHULA VISTA,04/30/2021 - 04/30/2029,True,130,2750,0.05,377,1777,0.21,943,1911,0.49,4716,4667,1.01,chula vista


## CA DOF population & housing estimates (E-5)

In [119]:
DOF_RAW_PATH = RAW_DIR.parent / "dof" / "e5_population_housing.xlsx"
DOF_SHEET_NAME = f"E5CityCounty{TARGET_YEAR}"

dof_raw = pd.read_excel(DOF_RAW_PATH, sheet_name=DOF_SHEET_NAME, header=3)
dof_raw = dof_raw.rename(columns={"County/City/State": "name"})
dof_raw["name"] = dof_raw["name"].astype(str).str.strip()

# The raw sheet has two columns both literally named "Total" - population
# total and housing-unit total. pandas auto-disambiguates the second one
# to "Total.1" on read; renamed here to something unambiguous.
dof_raw = dof_raw.rename(columns={"Total": "population_total", "Total.1": "housing_units_total"})

dof_raw["is_county_header"] = dof_raw["population_total"].isna() & dof_raw["name"].str.contains("County", na=False)
dof_raw["county"] = dof_raw["name"].where(dof_raw["is_county_header"]).ffill()

sd_dof = dof_raw[
    (dof_raw["county"] == "San Diego County")
    & (~dof_raw["is_county_header"])
    & (dof_raw["population_total"].notna())
    & (~dof_raw["name"].isin(["Incorporated", "County Total"]))
].copy()

sd_dof["jur_clean"] = sd_dof["name"].map(
    lambda n: normalize_jurisdiction(COUNTY_JURISDICTION_NAME) if n.strip() == "Unincorporated" else normalize_jurisdiction(n)
)
sd_dof["year"] = TARGET_YEAR

sd_dof["vacant_units"] = sd_dof["housing_units_total"] - sd_dof["Occupied"]
sd_dof["single_family_units"] = sd_dof["Single Detached"] + sd_dof["Single Attached"]
sd_dof["multifamily_units"] = sd_dof["Two to Four"] + sd_dof["Five Plus"]
sd_dof["mobile_home_units"] = sd_dof["Mobile Homes"]

structure_sum = sd_dof["single_family_units"] + sd_dof["multifamily_units"] + sd_dof["mobile_home_units"]
assert (structure_sum == sd_dof["housing_units_total"]).all(), "Structure-type columns do not sum to housing_units_total"

print("SD rows:", len(sd_dof))
missing = sd_jur_keys - set(sd_dof["jur_clean"].unique())
if missing:
    print("Not found:", sorted(missing))
sd_dof[[
    "name", "population_total", "housing_units_total", "Occupied", "vacant_units",
    "single_family_units", "multifamily_units", "mobile_home_units",
]]


SD rows: 19


,name,population_total,housing_units_total,Occupied,vacant_units,single_family_units,multifamily_units,mobile_home_units
615,Carlsbad,116022.0,48888.0,45710.0,3178.0,33861.0,13817.0,1210.0
616,Chula Vista,281850.0,91485.0,88373.0,3112.0,56792.0,30800.0,3893.0
617,Coronado,22687.0,9646.0,7438.0,2208.0,5463.0,4180.0,3.0
618,Del Mar,3937.0,2641.0,1952.0,689.0,1903.0,738.0,0.0
619,El Cajon,105449.0,37011.0,35686.0,1325.0,17276.0,17852.0,1883.0
620,Encinitas,62392.0,27118.0,25046.0,2072.0,20958.0,5522.0,638.0
621,Escondido,151932.0,50883.0,49041.0,1842.0,29260.0,17918.0,3705.0
622,Imperial Beach,26362.0,10269.0,9499.0,770.0,4934.0,5033.0,302.0
623,La Mesa,61863.0,26851.0,25814.0,1037.0,14068.0,12615.0,168.0
624,Lemon Grove,28445.0,9812.0,9462.0,350.0,7310.0,2424.0,78.0


## Census ACS (2020-2024 5-year estimates)

In [120]:
from getpass import getpass

CALIFORNIA_STATE_FIPS = "06"
SAN_DIEGO_COUNTY_FIPS = "073"

CENSUS_API_KEY = getpass("Census API key: ").strip()
if not CENSUS_API_KEY:
    raise ValueError("A Census API key is required.")

ACS_VARS = {
    "B01003_001E": "population_total",
    "B25001_001E": "housing_units_total",
    "B25003_002E": "owner_occupied",
    "B25003_003E": "renter_occupied",
}

acs_url = f"https://api.census.gov/data/{ACS_DATA_YEAR}/acs/acs5"
params = {
    "get": ",".join(["NAME", *ACS_VARS.keys()]),
    "for": "place:*",
    "in": f"state:{CALIFORNIA_STATE_FIPS}",
    "key": CENSUS_API_KEY,
}

acs_raw = get_json(acs_url, params=params)
acs_df = pd.DataFrame(acs_raw[1:], columns=acs_raw[0]).rename(columns=ACS_VARS)
acs_df["jur_clean"] = acs_df["NAME"].str.replace(r" city, California$", "", regex=True).map(normalize_jurisdiction)

sd_acs = acs_df[acs_df["jur_clean"].isin(sd_jur_keys)].copy()
print(f"SD rows ({ACS_VINTAGE_LABEL}):", len(sd_acs))
sd_acs.head()


Census API key:  ········


SD rows (2020-2024): 18


,NAME,population_total,housing_units_total,owner_occupied,renter_occupied,state,place,jur_clean
218,"Carlsbad city, California",114373,47314,27688,16350,06,11194,carlsbad
271,"Chula Vista city, California",276375,90273,51281,34429,06,13392,chula vista
315,"Coronado city, California",19015,9896,3991,3312,06,16378,coronado
364,"Del Mar city, California",3903,2550,987,868,06,18506,del mar
439,"El Cajon city, California",104449,35185,14066,19824,06,21712,el cajon


## Summary

In [121]:
# City of SD permits is intentionally not included here - it's an
# optional appendix section that runs later, not part of the core pipeline.
loaded = {
    "APR (permits/completions)": sd_apr_target_year if "sd_apr_target_year" in dir() else pd.DataFrame(),
    "RHNA6 (targets)": sd_rhna6 if "sd_rhna6" in dir() else pd.DataFrame(),
    "DOF (population/housing)": sd_dof if "sd_dof" in dir() else pd.DataFrame(),
    "ACS": sd_acs if "sd_acs" in dir() else pd.DataFrame(),
}
for name, df in loaded.items():
    if df.empty:
        print(f"{name:30s} not loaded")
    else:
        n_missing = (df.isna().sum() > 0).sum()
        print(f"{name:30s} {df.shape[0]} rows, {df.shape[1]} cols, {n_missing} cols with missing values")


APR (permits/completions)      7460 rows, 88 cols, 12 cols with missing values
RHNA6 (targets)                19 rows, 16 cols, 0 cols with missing values
DOF (population/housing)       19 rows, 21 cols, 0 cols with missing values
ACS                            18 rows, 8 cols, 0 cols with missing values


## Housing production categories & housing stock fields

In [122]:
production_categories = pd.DataFrame([
    {"category": "Application submitted", "income_tier": "Acutely Low through Above Moderate (6 tiers)", "source": "APR Table A", "field": "ACUTELY_LOW_INCOME_DR/_NDR ... ABOVE_MOD_INCOME (same 11-column pattern as Table A2's entitlement section)"},
    {"category": "Entitlement", "income_tier": "all tiers", "source": "APR Table A2", "field": "ENT_APPROVE_DT1, NO_ENTITLEMENTS"},
    {"category": "Building permit", "income_tier": "Acutely Low", "source": "APR Table A2", "field": "BP_ACUTELY_LOW_INCOME_DR / _NDR"},
    {"category": "Building permit", "income_tier": "Extremely Low", "source": "APR Table A2", "field": "BP_EXTREMELY_LOW_INCOME_DR / _NDR"},
    {"category": "Building permit", "income_tier": "Very Low", "source": "APR Table A2", "field": "BP_VLOW_INCOME_DR / _NDR"},
    {"category": "Building permit", "income_tier": "Low", "source": "APR Table A2", "field": "BP_LOW_INCOME_DR / _NDR"},
    {"category": "Building permit", "income_tier": "Moderate", "source": "APR Table A2", "field": "BP_MOD_INCOME_DR / _NDR"},
    {"category": "Building permit", "income_tier": "Above Moderate", "source": "APR Table A2", "field": "BP_ABOVE_MOD_INCOME"},
    {"category": "Completion (certificate of occupancy)", "income_tier": "Acutely Low", "source": "APR Table A2", "field": "CO_ACUTELY_LOW_INCOME_DR / _NDR"},
    {"category": "Completion (certificate of occupancy)", "income_tier": "Extremely Low", "source": "APR Table A2", "field": "CO_EXTREMELY_LOW_INCOME_DR / _NDR"},
    {"category": "Completion (certificate of occupancy)", "income_tier": "Very Low", "source": "APR Table A2", "field": "CO_VLOW_INCOME_DR / _NDR"},
    {"category": "Completion (certificate of occupancy)", "income_tier": "Low", "source": "APR Table A2", "field": "CO_LOW_INCOME_DR / _NDR"},
    {"category": "Completion (certificate of occupancy)", "income_tier": "Moderate", "source": "APR Table A2", "field": "CO_MOD_INCOME_DR / _NDR"},
    {"category": "Completion (certificate of occupancy)", "income_tier": "Above Moderate", "source": "APR Table A2", "field": "CO_ABOVE_MOD_INCOME"},
    {"category": "RHNA target", "income_tier": "VLI / LI / Moderate / Above Moderate", "source": "RHNA6 progress", "field": "RHNA VLI, RHNA LI, RHNA MOD, RHNA ABOVE MOD"},
    {"category": "RHNA progress", "income_tier": "VLI / LI / Moderate / Above Moderate", "source": "RHNA6 progress", "field": "VLI UNITS, LI UNITS, MOD UNITS, ABOVE MOD UNITS"},
    {"category": "City permit approval", "income_tier": "Extremely Low / Very Low / Low / Moderate / Above Moderate", "source": "City of SD permits", "field": "APPROVAL_DU_EXTREMELY_LOW ... APPROVAL_DU_ABOVE_MODERATE"},
    {"category": "ADU", "income_tier": "n/a", "source": "City of SD permits", "field": "APPROVAL_ADU_TOTAL (plus per-tier APPROVAL_ADU_* columns)"},
    {"category": "JADU", "income_tier": "n/a", "source": "City of SD permits", "field": "APPROVAL_JADU_TOTAL (plus per-tier APPROVAL_JADU_* columns)"},
    {"category": "Preservation (existing affordable units retained)", "income_tier": "n/a", "source": "APR Table F (not yet loaded in this notebook)", "field": "see sd_apr_f_preservation_city_year.csv in dashboard prototype"},
])
production_categories.to_csv(DOCS_DIR / "housing_production_categories.csv", index=False)
production_categories


,category,income_tier,source,field
0,Application submitted,Acutely Low through Above Moderate (6 tiers),APR Table A,ACUTELY_LOW_INCOME_DR/_NDR ... ABOVE_MOD_INCOM...
1,Entitlement,all tiers,APR Table A2,"ENT_APPROVE_DT1, NO_ENTITLEMENTS"
2,Building permit,Acutely Low,APR Table A2,BP_ACUTELY_LOW_INCOME_DR / _NDR
3,Building permit,Extremely Low,APR Table A2,BP_EXTREMELY_LOW_INCOME_DR / _NDR
4,Building permit,Very Low,APR Table A2,BP_VLOW_INCOME_DR / _NDR
5,Building permit,Low,APR Table A2,BP_LOW_INCOME_DR / _NDR
6,Building permit,Moderate,APR Table A2,BP_MOD_INCOME_DR / _NDR
7,Building permit,Above Moderate,APR Table A2,BP_ABOVE_MOD_INCOME
8,Completion (certificate of occupancy),Acutely Low,APR Table A2,CO_ACUTELY_LOW_INCOME_DR / _NDR
9,Completion (certificate of occupancy),Extremely Low,APR Table A2,CO_EXTREMELY_LOW_INCOME_DR / _NDR


In [123]:
stock_fields = pd.DataFrame([
    {"field": "Total population", "source": "DOF E-5", "column": "population_total (raw: \"Total\", disambiguated from housing_units_total)"},
    {"field": "Household population", "source": "DOF E-5", "column": "Household"},
    {"field": "Group quarters population", "source": "DOF E-5", "column": "Group Quarters"},
    {"field": "Total housing units", "source": "DOF E-5", "column": "housing_units_total (raw: \"Total.1\")"},
    {"field": "Occupied units", "source": "DOF E-5", "column": "occupied_units (raw: \"Occupied\")"},
    {"field": "Vacant units", "source": "DOF E-5", "column": "vacant_units (computed: housing_units_total - Occupied)"},
    {"field": "Single-family units", "source": "DOF E-5", "column": "single_family_units (computed: Single Detached + Single Attached)"},
    {"field": "Multifamily units", "source": "DOF E-5", "column": "multifamily_units (computed: Two to Four + Five Plus)"},
    {"field": "Mobile home units", "source": "DOF E-5", "column": "mobile_home_units (raw: Mobile Homes)"},
    {"field": "Single detached units (detail)", "source": "DOF E-5", "column": "Single Detached"},
    {"field": "Single attached units (detail)", "source": "DOF E-5", "column": "Single Attached"},
    {"field": "2-4 unit buildings (detail)", "source": "DOF E-5", "column": "Two to Four"},
    {"field": "5+ unit buildings (detail)", "source": "DOF E-5", "column": "Five Plus"},
    {"field": "Vacancy rate", "source": "DOF E-5", "column": "Vacancy Rate"},
    {"field": "Persons per household", "source": "DOF E-5", "column": "Persons per Household"},
    {"field": "Total population", "source": "ACS 5-year", "column": "B01003_001E (population_total)"},
    {"field": "Total housing units", "source": "ACS 5-year", "column": "B25001_001E (housing_units_total)"},
    {"field": "Owner-occupied units", "source": "ACS 5-year", "column": "B25003_002E (owner_occupied)"},
    {"field": "Renter-occupied units", "source": "ACS 5-year", "column": "B25003_003E (renter_occupied)"},
])
stock_fields.to_csv(DOCS_DIR / "housing_stock_fields.csv", index=False)
stock_fields


,field,source,column
0,Total population,DOF E-5,"population_total (raw: ""Total"", disambiguated ..."
1,Household population,DOF E-5,Household
2,Group quarters population,DOF E-5,Group Quarters
3,Total housing units,DOF E-5,"housing_units_total (raw: ""Total.1"")"
4,Occupied units,DOF E-5,"occupied_units (raw: ""Occupied"")"
5,Vacant units,DOF E-5,vacant_units (computed: housing_units_total - ...
6,Single-family units,DOF E-5,single_family_units (computed: Single Detached...
7,Multifamily units,DOF E-5,multifamily_units (computed: Two to Four + Fiv...
8,Mobile home units,DOF E-5,mobile_home_units (raw: Mobile Homes)
9,Single detached units (detail),DOF E-5,Single Detached


**Notes on coverage:**

- DOF and ACS both report total population/housing units, but DOF breaks
  housing units out by structure type (single/multi/mobile) while ACS
  breaks out tenure (owner/renter) instead - they're complementary, not
  duplicates, unlike the APR/City-permits overlap found below.
- The APR field names use `VLOW` for "Very Low" while RHNA6 uses `VLI` -
  same tier, different abbreviation between the two datasets.
- DR/NDR suffixes on APR income columns = Deed Restricted / Non-Deed
  Restricted (whether the affordability requirement is legally recorded
  on the property).
- Preservation (Table F) is not yet loaded into this notebook - it exists
  as a processed file in the dashboard prototype repo but isn't part of
  this workstream's live pulls yet.


## Jurisdiction-year RHNA and housing production dataset

In [124]:
# RHNA6 has no year column (cumulative cycle-to-date, not annual), so its
# target/progress numbers get joined onto each jurisdiction as static
# context rather than matched by year.
#
# Kept broken out by income category (not just totals).
RHNA_TIERS = {
    "very_low": ("RHNA VLI", "VLI UNITS"),
    "low": ("RHNA LI", "LI UNITS"),
    "moderate": ("RHNA MOD", "MOD UNITS"),
    "above_moderate": ("RHNA ABOVE MOD", "ABOVE MOD UNITS"),
}

rhna_summary = sd_rhna6.copy()
tier_cols = ["jur_clean"]

for tier, (target_col, reported_col) in RHNA_TIERS.items():
    rhna_summary[f"rhna_target_{tier}"] = rhna_summary[target_col]
    rhna_summary[f"rhna_reported_{tier}"] = rhna_summary[reported_col]
    # Clipped at 0 - overachieving one tier doesn't offset a shortfall in
    # another; each income tier is a separate obligation, not a shared pool.
    rhna_summary[f"rhna_remaining_{tier}"] = (
        rhna_summary[f"rhna_target_{tier}"] - rhna_summary[f"rhna_reported_{tier}"]
    ).clip(lower=0)
    rhna_summary[f"rhna_pct_achieved_{tier}"] = rhna_summary[f"rhna_reported_{tier}"] / rhna_summary[f"rhna_target_{tier}"]
    tier_cols += [
        f"rhna_target_{tier}", f"rhna_reported_{tier}",
        f"rhna_remaining_{tier}", f"rhna_pct_achieved_{tier}",
    ]

rhna_summary["rhna_target_total"] = rhna_summary[["RHNA VLI", "RHNA LI", "RHNA MOD", "RHNA ABOVE MOD"]].sum(axis=1)
rhna_summary["rhna_units_reported_total"] = rhna_summary[["VLI UNITS", "LI UNITS", "MOD UNITS", "ABOVE MOD UNITS"]].sum(axis=1)
rhna_summary["rhna_remaining_total"] = rhna_summary[[f"rhna_remaining_{t}" for t in RHNA_TIERS]].sum(axis=1)
rhna_summary["rhna_pct_achieved"] = rhna_summary["rhna_units_reported_total"] / rhna_summary["rhna_target_total"]
tier_cols += ["rhna_target_total", "rhna_units_reported_total", "rhna_remaining_total", "rhna_pct_achieved"]

rhna_summary = rhna_summary[tier_cols]

dof_summary = sd_dof.rename(columns={"Household": "population_household"})[[
    "jur_clean", "population_total", "population_household",
    "housing_units_total", "Occupied", "vacant_units",
    "single_family_units", "multifamily_units", "mobile_home_units",
]].rename(columns={"Occupied": "occupied_units"})

jurisdiction_year_dataset = (
    production_by_year
    .merge(applications_by_jurisdiction, on="jur_clean", how="left")
    .merge(rhna_summary, on="jur_clean", how="left")
    .merge(dof_summary, on="jur_clean", how="left")
)

print(jurisdiction_year_dataset.shape)
jurisdiction_year_dataset


(19, 60)


,jur_clean,year,ent_units_total,bp_units_total,co_units_total,ent_affordable_total,bp_affordable_total,co_affordable_total,project_rows,ent_very_low_total,...,rhna_remaining_total,rhna_pct_achieved,population_total,population_household,housing_units_total,occupied_units,vacant_units,single_family_units,multifamily_units,mobile_home_units
0,carlsbad,2025,202,343,719,13,36,141,428,11,...,2326,0.399432,116022.0,115034.0,48888.0,45710.0,3178.0,33861.0,13817.0,1210.0
1,chula vista,2025,1323,723,1428,0,224,201,663,0,...,4988,0.555245,281850.0,280211.0,91485.0,88373.0,3112.0,56792.0,30800.0,3893.0
2,coronado,2025,27,24,36,0,0,0,54,0,...,708,0.223684,22687.0,17283.0,9646.0,7438.0,2208.0,5463.0,4180.0,3.0
3,del mar,2025,15,14,17,12,11,10,43,0,...,101,0.680982,3937.0,3937.0,2641.0,1952.0,689.0,1903.0,738.0,0.0
4,el cajon,2025,99,210,235,40,83,123,164,0,...,2495,0.239329,105449.0,102949.0,37011.0,35686.0,1325.0,17276.0,17852.0,1883.0
5,encinitas,2025,277,187,228,40,33,43,461,0,...,832,0.932432,62392.0,61835.0,27118.0,25046.0,2072.0,20958.0,5522.0,638.0
6,escondido,2025,366,514,604,234,246,66,396,119,...,7503,0.219007,151932.0,149374.0,50883.0,49041.0,1842.0,29260.0,17918.0,3705.0
7,imperial beach,2025,45,71,13,0,0,0,86,0,...,1116,0.160271,26362.0,25995.0,10269.0,9499.0,770.0,4934.0,5033.0,302.0
8,la mesa,2025,78,254,249,11,99,215,223,0,...,2741,0.278114,61863.0,61169.0,26851.0,25814.0,1037.0,14068.0,12615.0,168.0
9,lemon grove,2025,1,32,61,0,6,8,77,0,...,1010,0.256806,28445.0,28080.0,9812.0,9462.0,350.0,7310.0,2424.0,78.0


In [125]:
missing = sd_jur_keys - set(jurisdiction_year_dataset["jur_clean"].unique())
if missing:
    print("Jurisdictions missing from the combined table:", sorted(missing))
else:
    print("All 18 cities + County present.")

null_counts = jurisdiction_year_dataset.isna().sum()
null_counts[null_counts > 0]


All 18 cities + County present.


Series([], dtype: int64)

## San Diego regional total

Separate object, not a 20th row in `jurisdiction_year_dataset` - keeps
countywide and jurisdiction-level results distinct. Sums all 18 cities
plus the unincorporated county.

In [126]:
assert len(jurisdiction_year_dataset) == 19, (
    f"Expected 18 cities + unincorporated county = 19 rows, got {len(jurisdiction_year_dataset)}"
)

RHNA_TIER_NAMES = ["very_low", "low", "moderate", "above_moderate"]
PRODUCTION_TIER_NAMES = ["very_low", "low", "moderate", "above_moderate"]

SUM_COLS = [
    "application_units_total", "ent_units_total", "bp_units_total", "co_units_total",
    "application_affordable_total", "ent_affordable_total", "bp_affordable_total", "co_affordable_total",
    "project_rows", "rhna_target_total", "rhna_units_reported_total", "rhna_remaining_total",
    "population_total", "population_household",
    "housing_units_total", "occupied_units", "vacant_units",
    "single_family_units", "multifamily_units", "mobile_home_units",
]
SUM_COLS += [f"rhna_target_{t}" for t in RHNA_TIER_NAMES]
SUM_COLS += [f"rhna_reported_{t}" for t in RHNA_TIER_NAMES]
SUM_COLS += [f"rhna_remaining_{t}" for t in RHNA_TIER_NAMES]
SUM_COLS += [f"bp_{t}_total" for t in PRODUCTION_TIER_NAMES]
SUM_COLS += [f"co_{t}_total" for t in PRODUCTION_TIER_NAMES]

sd_region_total = jurisdiction_year_dataset[SUM_COLS].sum().to_frame().T
sd_region_total.insert(0, "jurisdiction", "San Diego Region (18 cities + Unincorporated County)")
sd_region_total.insert(1, "year", TARGET_YEAR)
sd_region_total.insert(2, "jurisdictions_included", len(jurisdiction_year_dataset))

# Share/percentage fields recalculated at the regional level, not summed
# or averaged directly.
sd_region_total["application_affordable_share"] = sd_region_total["application_affordable_total"] / sd_region_total["application_units_total"]
sd_region_total["ent_affordable_share"] = sd_region_total["ent_affordable_total"] / sd_region_total["ent_units_total"]
sd_region_total["bp_affordable_share"] = sd_region_total["bp_affordable_total"] / sd_region_total["bp_units_total"]
sd_region_total["co_affordable_share"] = sd_region_total["co_affordable_total"] / sd_region_total["co_units_total"]
sd_region_total["rhna_pct_achieved"] = sd_region_total["rhna_units_reported_total"] / sd_region_total["rhna_target_total"]

for tier in RHNA_TIER_NAMES:
    sd_region_total[f"rhna_pct_achieved_{tier}"] = (
        sd_region_total[f"rhna_reported_{tier}"] / sd_region_total[f"rhna_target_{tier}"]
    )

sd_region_total


,jurisdiction,year,jurisdictions_included,application_units_total,ent_units_total,bp_units_total,co_units_total,application_affordable_total,ent_affordable_total,bp_affordable_total,...,co_above_moderate_total,application_affordable_share,ent_affordable_share,bp_affordable_share,co_affordable_share,rhna_pct_achieved,rhna_pct_achieved_very_low,rhna_pct_achieved_low,rhna_pct_achieved_moderate,rhna_pct_achieved_above_moderate
0,San Diego Region (18 cities + Unincorporated C...,2025,19,15838.0,6257.0,13221.0,6337.0,3727.0,1042.0,3717.0,...,4784.0,0.23532,0.166533,0.281144,0.245069,0.37941,0.094302,0.23709,0.216015,0.663237


In [127]:
check = jurisdiction_year_dataset["bp_units_total"].sum() == sd_region_total["bp_units_total"].iloc[0]
print("Regional bp_units_total matches sum of jurisdiction rows:", check)

region_output_path = PROCESSED_DIR / f"rhna_housing_production_{TARGET_YEAR}_regional_total.csv"
sd_region_total.to_csv(region_output_path, index=False)
print("Saved:", region_output_path)


Regional bp_units_total matches sum of jurisdiction rows: True
Saved: /Users/ice/Documents/GitHub/chpd-dashboard-data-validation/van's work/data/processed/rhna_housing_production_2025_regional_total.csv


## Power BI-ready export (long format)

Wide format (one column per tier per stage) doesn't work well in Power BI
- it wants one row per (jurisdiction, year, income category, development
stage) with a single value column. Reshapes the multi-year tables
(`production_history` for entitlement/permit/completion, `applications_history`
for applications) into that structure, so a year filter in Power BI works
across all 4 stages.

In [128]:
STAGE_SOURCE_MAP = {
    "Application": (applications_history, "application"),
    "Entitlement": (production_history, "ent"),
    "Building Permit": (production_history, "bp"),
    "Completion": (production_history, "co"),
}
INCOME_TIERS = ["very_low", "low", "moderate", "above_moderate"]

long_rows = []
for stage_label, (source_df, prefix) in STAGE_SOURCE_MAP.items():
    for tier in INCOME_TIERS:
        col = f"{prefix}_{tier}_total"
        if col not in source_df.columns:
            continue
        chunk = source_df[["jur_clean", "year"]].copy()
        chunk["development_stage"] = stage_label
        chunk["income_category"] = tier
        chunk["units"] = source_df[col]
        long_rows.append(chunk)

powerbi_long = pd.concat(long_rows, ignore_index=True)
powerbi_long = powerbi_long.rename(columns={"jur_clean": "jurisdiction"})
powerbi_long = powerbi_long.sort_values(["jurisdiction", "year", "development_stage", "income_category"]).reset_index(drop=True)

print(powerbi_long.shape)
print("Years present:", sorted(powerbi_long["year"].unique()))
powerbi_long.head(20)


(2432, 5)
Years present: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


,jurisdiction,year,development_stage,income_category,units
0,carlsbad,2018,Application,above_moderate,313.0
1,carlsbad,2018,Application,low,43.0
2,carlsbad,2018,Application,moderate,6.0
3,carlsbad,2018,Application,very_low,7.0
4,carlsbad,2018,Building Permit,above_moderate,210.0
5,carlsbad,2018,Building Permit,low,5.0
6,carlsbad,2018,Building Permit,moderate,28.0
7,carlsbad,2018,Building Permit,very_low,0.0
8,carlsbad,2018,Completion,above_moderate,193.0
9,carlsbad,2018,Completion,low,4.0


In [129]:
expected_rows = 19 * len(STAGE_SOURCE_MAP) * len(INCOME_TIERS) * len(APR_YEARS)
assert len(powerbi_long) == expected_rows, f"Expected {expected_rows} rows, got {len(powerbi_long)}"

check = powerbi_long[powerbi_long["year"] == TARGET_YEAR].groupby(
    ["jurisdiction", "development_stage"]
)["units"].sum().reset_index()
bp_check = check[check["development_stage"] == "Building Permit"].set_index("jurisdiction")["units"]
original_bp = production_history[production_history["year"] == TARGET_YEAR].set_index("jur_clean")["bp_units_total"]
assert (bp_check.reindex(original_bp.index) == original_bp).all(), "Long-format BP totals do not match original wide totals"

print("Row count and reshape checks passed:", len(powerbi_long), "rows across", len(APR_YEARS), "years")

powerbi_path = PROCESSED_DIR / f"powerbi_rhna_production_{APR_START_YEAR}_{TARGET_YEAR}_long.csv"
powerbi_long.to_csv(powerbi_path, index=False)
print("Saved:", powerbi_path)


Row count and reshape checks passed: 2432 rows across 8 years
Saved: /Users/ice/Documents/GitHub/chpd-dashboard-data-validation/van's work/data/processed/powerbi_rhna_production_2018_2025_long.csv


**Coverage:** spans 2018-2025 for all 4 stages. Check the year-coverage
output above (applications and permits/completions sections) for any year
with fewer than 19 jurisdictions before filtering to it in Power BI.

## Metric dictionary

In [130]:
metric_dictionary = pd.DataFrame([
    {
        "output_metric": "application_units_total",
        "source_table": "HCD APR Table A",
        "source_variables": "Sum of unprefixed *_INCOME_* columns, all APPLICATION_STATUS values included",
        "definition": "Total housing units in applications submitted, all income tiers, per jurisdiction-year. Counts all applications regardless of approval status - this is 'submitted', not 'approved'. Cross-checked against Table A's own TOT_PROPOSED_UNITS field",
        "source_year": TARGET_YEAR, "download_date": DOWNLOAD_DATE,
        "geographic_level": "Jurisdiction (18 cities + unincorporated county)",
        "unit_of_measurement": "Housing units (count)",
        "limitations": "Small gaps vs. TOT_PROPOSED_UNITS for 5 of 19 jurisdictions - see data_quality_limitations.csv",
    },
    {
        "output_metric": "ent_units_total",
        "source_table": "HCD APR Table A2",
        "source_variables": "Sum of unprefixed *_INCOME_* columns (not BP_ or CO_ prefixed)",
        "definition": "Total housing units with a planning entitlement approved, all income tiers, per jurisdiction-year. Kept fully separate from bp_units_total and co_units_total - never summed together",
        "source_year": TARGET_YEAR, "download_date": DOWNLOAD_DATE,
        "geographic_level": "Jurisdiction (18 cities + unincorporated county)",
        "unit_of_measurement": "Housing units (count)",
        "limitations": "Matches HCD's own auto-populated NO_ENTITLEMENTS total exactly for all 19 jurisdictions",
    },
    {
        "output_metric": "bp_units_total / bp_<tier>_total",
        "source_table": "HCD APR Table A2",
        "source_variables": "Sum of BP_*_INCOME columns, kept both as an overall total and broken out per income tier (very_low, low, moderate, above_moderate)",
        "definition": "Total housing units with a building permit issued, per jurisdiction-year. Per-tier columns sum back to the overall total exactly (enforced by an assertion in the notebook)",
        "source_year": f"2018-{TARGET_YEAR} (historical export); {TARGET_YEAR} (jurisdiction-year table)",
        "download_date": DOWNLOAD_DATE,
        "geographic_level": "Jurisdiction (18 cities + unincorporated county)",
        "unit_of_measurement": "Housing units (count)",
        "limitations": "Self-reported by jurisdictions to HCD, not independently verified",
    },
    {
        "output_metric": "co_units_total / co_<tier>_total",
        "source_table": "HCD APR Table A2",
        "source_variables": "Sum of CO_*_INCOME columns, kept both as an overall total and broken out per income tier (very_low, low, moderate, above_moderate)",
        "definition": "Total housing units with a certificate of occupancy (completed), per jurisdiction-year. Per-tier columns sum back to the overall total exactly (enforced by an assertion in the notebook)",
        "source_year": f"2018-{TARGET_YEAR} (historical export); {TARGET_YEAR} (jurisdiction-year table)",
        "download_date": DOWNLOAD_DATE,
        "geographic_level": "Jurisdiction (18 cities + unincorporated county)",
        "unit_of_measurement": "Housing units (count)",
        "limitations": "Self-reported by jurisdictions to HCD, not independently verified",
    },
    {
        "output_metric": "bp_affordable_share",
        "source_table": "HCD APR Table A2",
        "source_variables": "bp_affordable_total / bp_units_total",
        "definition": "Share of permitted units in VLI, LI, or Moderate income categories (excludes Above Moderate)",
        "source_year": TARGET_YEAR, "download_date": DOWNLOAD_DATE,
        "geographic_level": "Jurisdiction (18 cities + unincorporated county)",
        "unit_of_measurement": "Percent (0-1 share)",
        "limitations": "NaN when bp_units_total is 0 that year - zero permits, not missing data",
    },
    {
        "output_metric": "co_affordable_share",
        "source_table": "HCD APR Table A2",
        "source_variables": "co_affordable_total / co_units_total",
        "definition": "Share of completed units in VLI, LI, or Moderate income categories (excludes Above Moderate)",
        "source_year": TARGET_YEAR, "download_date": DOWNLOAD_DATE,
        "geographic_level": "Jurisdiction (18 cities + unincorporated county)",
        "unit_of_measurement": "Percent (0-1 share)",
        "limitations": "NaN when co_units_total is 0 that year - zero completions, not missing data",
    },
    {
        "output_metric": "rhna_target_<tier>",
        "source_table": "HCD RHNA 6th Cycle Progress Report (Table B)",
        "source_variables": "RHNA VLI, RHNA LI, RHNA MOD, RHNA ABOVE MOD",
        "definition": "Assigned RHNA target units, kept separate per income tier (very_low, low, moderate, above_moderate) for the 6th Cycle planning period, per jurisdiction - plus rhna_target_total for the summed value",
        "source_year": "6th Cycle (2021-2029), adopted allocation, does not change during the cycle",
        "download_date": DOWNLOAD_DATE,
        "geographic_level": "Jurisdiction (18 cities + unincorporated county) and regional total",
        "unit_of_measurement": "Housing units (count)",
        "limitations": "Validated exact match against HCD/SANDAG published figures",
    },
    {
        "output_metric": "rhna_reported_<tier> / rhna_pct_achieved_<tier> / rhna_remaining_<tier>",
        "source_table": "HCD RHNA 6th Cycle Progress Report (Table B)",
        "source_variables": "VLI UNITS, LI UNITS, MOD UNITS, ABOVE MOD UNITS",
        "definition": "BASED ON BUILDING PERMITS ISSUED, not completed units - this is HCD's own RHNA-credit methodology. Kept separate per income tier, per jurisdiction. Cumulative cycle-to-date, not year-by-year. Do not conflate with co_units_total (completions)",
        "source_year": f"Cumulative, 6th Cycle to date (as of {TARGET_YEAR} data pull)",
        "download_date": DOWNLOAD_DATE,
        "geographic_level": "Jurisdiction (18 cities + unincorporated county) and regional total",
        "unit_of_measurement": "Housing units (count); Percent for rhna_pct_achieved_<tier>",
        "limitations": "Cumulative cycle-to-date, not annual",
    },
    {
        "output_metric": "sd_permit_du_by_tier",
        "source_table": "City of San Diego Development Permits (Active + Closed approvals)",
        "source_variables": "APPROVAL_DU_EXTREMELY_LOW/VERY_LOW/LOW/MODERATE/ABOVE_MODERATE",
        "definition": "Dwelling units per approval, broken out by income tier, City of San Diego only. Separate ADU (APPROVAL_ADU_*) and JADU (APPROVAL_JADU_*) columns exist alongside standard units",
        "source_year": TARGET_YEAR, "download_date": DOWNLOAD_DATE,
        "geographic_level": "City of San Diego only",
        "unit_of_measurement": "Housing units (count)",
        "limitations": "Overlaps with APR for San Diego - do not sum with bp_units_total. Appendix data, not in core exports",
    },
    {
        "output_metric": "sd_permit_stage",
        "source_table": "City of San Diego Development Permits (Active + Closed approvals)",
        "source_variables": "APPROVAL_ISSUE_DATE, APPROVAL_CLOSE_DATE, approval_status",
        "definition": "This dataset tracks permit issuance and closure only - it has no certificate-of-occupancy / completion field equivalent to APR's CO_* columns",
        "source_year": TARGET_YEAR, "download_date": DOWNLOAD_DATE,
        "geographic_level": "City of San Diego only",
        "unit_of_measurement": "Date / status (not a count)",
        "limitations": "'Closed' can mean finaled, expired, or withdrawn - not necessarily completed construction",
    },
])
metric_dictionary.to_csv(DOCS_DIR / "rhna_housing_production_metric_dictionary.csv", index=False)
metric_dictionary


,output_metric,source_table,source_variables,definition,source_year,download_date,geographic_level,unit_of_measurement,limitations
0,application_units_total,HCD APR Table A,"Sum of unprefixed *_INCOME_* columns, all APPL...","Total housing units in applications submitted,...",2025,2026-08-11,Jurisdiction (18 cities + unincorporated county),Housing units (count),Small gaps vs. TOT_PROPOSED_UNITS for 5 of 19 ...
1,ent_units_total,HCD APR Table A2,Sum of unprefixed *_INCOME_* columns (not BP_ ...,Total housing units with a planning entitlemen...,2025,2026-08-11,Jurisdiction (18 cities + unincorporated county),Housing units (count),Matches HCD's own auto-populated NO_ENTITLEMEN...
2,bp_units_total / bp_<tier>_total,HCD APR Table A2,"Sum of BP_*_INCOME columns, kept both as an ov...",Total housing units with a building permit iss...,2018-2025 (historical export); 2025 (jurisdict...,2026-08-11,Jurisdiction (18 cities + unincorporated county),Housing units (count),"Self-reported by jurisdictions to HCD, not ind..."
3,co_units_total / co_<tier>_total,HCD APR Table A2,"Sum of CO_*_INCOME columns, kept both as an ov...",Total housing units with a certificate of occu...,2018-2025 (historical export); 2025 (jurisdict...,2026-08-11,Jurisdiction (18 cities + unincorporated county),Housing units (count),"Self-reported by jurisdictions to HCD, not ind..."
4,bp_affordable_share,HCD APR Table A2,bp_affordable_total / bp_units_total,"Share of permitted units in VLI, LI, or Modera...",2025,2026-08-11,Jurisdiction (18 cities + unincorporated county),Percent (0-1 share),NaN when bp_units_total is 0 that year - zero ...
5,co_affordable_share,HCD APR Table A2,co_affordable_total / co_units_total,"Share of completed units in VLI, LI, or Modera...",2025,2026-08-11,Jurisdiction (18 cities + unincorporated county),Percent (0-1 share),NaN when co_units_total is 0 that year - zero ...
6,rhna_target_<tier>,HCD RHNA 6th Cycle Progress Report (Table B),"RHNA VLI, RHNA LI, RHNA MOD, RHNA ABOVE MOD","Assigned RHNA target units, kept separate per ...","6th Cycle (2021-2029), adopted allocation, doe...",2026-08-11,Jurisdiction (18 cities + unincorporated count...,Housing units (count),Validated exact match against HCD/SANDAG publi...
7,rhna_reported_<tier> / rhna_pct_achieved_<tier...,HCD RHNA 6th Cycle Progress Report (Table B),"VLI UNITS, LI UNITS, MOD UNITS, ABOVE MOD UNITS","BASED ON BUILDING PERMITS ISSUED, not complete...","Cumulative, 6th Cycle to date (as of 2025 data...",2026-08-11,Jurisdiction (18 cities + unincorporated count...,Housing units (count); Percent for rhna_pct_ac...,"Cumulative cycle-to-date, not annual"
8,sd_permit_du_by_tier,City of San Diego Development Permits (Active ...,APPROVAL_DU_EXTREMELY_LOW/VERY_LOW/LOW/MODERAT...,"Dwelling units per approval, broken out by inc...",2025,2026-08-11,City of San Diego only,Housing units (count),Overlaps with APR for San Diego - do not sum w...
9,sd_permit_stage,City of San Diego Development Permits (Active ...,"APPROVAL_ISSUE_DATE, APPROVAL_CLOSE_DATE, appr...",This dataset tracks permit issuance and closur...,2025,2026-08-11,City of San Diego only,Date / status (not a count),"'Closed' can mean finaled, expired, or withdra..."


## Export

In [131]:
output_path = PROCESSED_DIR / f"rhna_housing_production_{TARGET_YEAR}_by_jurisdiction.csv"
jurisdiction_year_dataset.to_csv(output_path, index=False)
print("Saved:", output_path)


Saved: /Users/ice/Documents/GitHub/chpd-dashboard-data-validation/van's work/data/processed/rhna_housing_production_2025_by_jurisdiction.csv


### Export metadata

Dataset-level metadata for each exported file - kept as its own record
rather than repeated on every row.

In [132]:
export_metadata = pd.DataFrame([
    {
        "file_name": f"rhna_housing_production_{TARGET_YEAR}_by_jurisdiction.csv",
        "source_year": TARGET_YEAR,
        "download_date": DOWNLOAD_DATE,
        "geographic_level": "Jurisdiction (18 cities + unincorporated San Diego County)",
        "unit_of_measurement": "Housing units (count); percent for share/pct_achieved fields",
        "limitations": "See data_quality_limitations.csv for full detail",
    },
    {
        "file_name": f"apr_production_{APR_START_YEAR}_{TARGET_YEAR}_by_jurisdiction_year.csv",
        "source_year": f"{APR_START_YEAR}-{TARGET_YEAR}",
        "download_date": DOWNLOAD_DATE,
        "geographic_level": "Jurisdiction (18 cities + unincorporated San Diego County), by year",
        "unit_of_measurement": "Housing units (count); percent for affordable-share fields",
        "limitations": "2018 (APR's first collection year) may have lower reporting completeness than later years",
    },
    {
        "file_name": f"rhna_housing_production_{TARGET_YEAR}_regional_total.csv",
        "source_year": TARGET_YEAR,
        "download_date": DOWNLOAD_DATE,
        "geographic_level": "Regional (all 18 cities + unincorporated San Diego County combined)",
        "unit_of_measurement": "Housing units (count); percent for share/pct_achieved fields",
        "limitations": "Percent fields recalculated at the regional level, not averaged from jurisdiction percentages",
    },
    {
        "file_name": f"powerbi_rhna_production_{APR_START_YEAR}_{TARGET_YEAR}_long.csv",
        "source_year": f"{APR_START_YEAR}-{TARGET_YEAR}",
        "download_date": DOWNLOAD_DATE,
        "geographic_level": "Jurisdiction (18 cities + unincorporated San Diego County)",
        "unit_of_measurement": "Housing units (count) - one row per jurisdiction/year/income category/development stage",
        "limitations": "Check per-year jurisdiction coverage before filtering - 2018 (APR's first year) may be less complete than later years",
    },
])
export_metadata.to_csv(DOCS_DIR / "export_metadata.csv", index=False)
export_metadata


,file_name,source_year,download_date,geographic_level,unit_of_measurement,limitations
0,rhna_housing_production_2025_by_jurisdiction.csv,2025,2026-08-11,Jurisdiction (18 cities + unincorporated San D...,Housing units (count); percent for share/pct_a...,See data_quality_limitations.csv for full detail
1,apr_production_2018_2025_by_jurisdiction_year.csv,2018-2025,2026-08-11,Jurisdiction (18 cities + unincorporated San D...,Housing units (count); percent for affordable-...,2018 (APR's first collection year) may have lo...
2,rhna_housing_production_2025_regional_total.csv,2025,2026-08-11,Regional (all 18 cities + unincorporated San D...,Housing units (count); percent for share/pct_a...,Percent fields recalculated at the regional le...
3,powerbi_rhna_production_2018_2025_long.csv,2018-2025,2026-08-11,Jurisdiction (18 cities + unincorporated San D...,Housing units (count) - one row per jurisdicti...,Check per-year jurisdiction coverage before fi...


## Validate against HCD's published RHNA results

Checks `rhna_target_total` against figures from official adopted planning
documents (SANDAG Board Resolution, adopted Housing Elements) - a
different publication channel than the CKAN dataset this notebook pulls
from.

In [133]:
PUBLISHED_RHNA = pd.DataFrame([
    {"jur_clean": "san diego", "published_rhna_target": 108036,
     "source": "City of San Diego adopted Housing Element 2021-2029"},
    {"jur_clean": "unincorporated san diego county", "published_rhna_target": 6700,
     "source": "County of San Diego General Plan Annual Progress Report"},
    {"jur_clean": "oceanside", "published_rhna_target": 5443,
     "source": "City of Oceanside Housing Element page"},
])

rhna_check = jurisdiction_year_dataset[["jur_clean", "rhna_target_total"]].merge(
    PUBLISHED_RHNA, on="jur_clean", how="inner"
)
rhna_check["diff"] = rhna_check["rhna_target_total"] - rhna_check["published_rhna_target"]
print("Jurisdiction-level check:")
display(rhna_check)

regional_published = 171685
regional_ours = sd_region_total["rhna_target_total"].iloc[0]
print(f"\nRegional total: ours={regional_ours:.0f}, published={regional_published}, "
      f"match={regional_ours == regional_published}")


Jurisdiction-level check:


,jur_clean,rhna_target_total,published_rhna_target,source,diff
0,oceanside,5443,5443,City of Oceanside Housing Element page,0
1,san diego,108036,108036,City of San Diego adopted Housing Element 2021...,0
2,unincorporated san diego county,6700,6700,County of San Diego General Plan Annual Progre...,0



Regional total: ours=171685, published=171685, match=True


**Result:** exact match, regional total + all 3 jurisdictions checked.
Validates `rhna_target_total` only - `rhna_reported_*` is cross-checked
separately against `NO_ENTITLEMENTS`/etc. in the APR section (HCD doesn't
publish a canonical "progress" number the way it does for targets).

## Validation against dashboard prototype

In [134]:
def find_sibling_repo(repo_name: str, search_depth: int = 3) -> Path | None:
    candidates = [ROOT, *ROOT.parents][:search_depth + 2]
    for base in candidates:
        match = base / repo_name
        if match.exists():
            return match
        if base.parent.exists():
            for sibling in base.parent.iterdir():
                if sibling.name == repo_name and sibling.is_dir():
                    return sibling
    return None


baseline_repo = find_sibling_repo("housing-dashboard-prototype")

if baseline_repo is None:
    print(
        "Could not find a local clone of housing-dashboard-prototype near this repo. "
        "Clone it into the same parent folder as this repo, then re-run this cell."
    )
else:
    BASELINE_PATH = baseline_repo / "data" / "processed" / "sd_apr_a2_city_year_supply.csv"
    print("Found baseline repo at:", baseline_repo)


Found baseline repo at: /Users/ice/Documents/GitHub/housing-dashboard-prototype


In [135]:
if baseline_repo is not None and BASELINE_PATH.exists():
    baseline = pd.read_csv(BASELINE_PATH)
    baseline["jur_clean"] = baseline["jur_clean"].str.lower()

    baseline_years = sorted(baseline["year"].unique())
    our_years = sorted(production_history["year"].dropna().unique())
    overlapping_years = sorted(set(baseline_years) & set(our_years))
    years_ahead = sorted(set(our_years) - set(baseline_years))

    if years_ahead:
        print(
            f"Baseline (dashboard prototype) does not yet include: {years_ahead}. "
            "This notebook's pull is ahead of the dashboard prototype for those years - "
            "excluded from the comparison below, not treated as mismatches."
        )

    if not overlapping_years:
        print("No overlapping years between this notebook's pull and the baseline - nothing to validate.")
    else:
        comparison = production_history.merge(
            baseline, on=["jur_clean", "year"], suffixes=("_fresh", "_baseline"), how="inner",
        )
        failed_checks = comparison[
            comparison["bp_units_total_fresh"] != comparison["bp_units_total_baseline"]
        ]
        print(f"Validated years: {overlapping_years}")
        print(f"Compared {len(comparison)} rows; {len(failed_checks)} mismatches.")
        display(failed_checks[["jur_clean", "year", "bp_units_total_fresh", "bp_units_total_baseline"]])
        comparison.to_csv(PROCESSED_DIR / "rhna_housing_production_validation_report.csv", index=False)
elif baseline_repo is not None:
    print(f"Repo found but expected file is missing: {BASELINE_PATH}")


Baseline (dashboard prototype) does not yet include: [np.int64(2025)]. This notebook's pull is ahead of the dashboard prototype for those years - excluded from the comparison below, not treated as mismatches.
Validated years: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Compared 126 rows; 9 mismatches.


,jur_clean,year,bp_units_total_fresh,bp_units_total_baseline
7,chula vista,2018,1777,1675.0
11,chula vista,2022,1185,1182.0
38,encinitas,2021,153,149.0
39,encinitas,2022,157,152.0
40,encinitas,2023,280,269.0
41,encinitas,2024,701,695.0
83,oceanside,2024,601,595.0
97,san diego,2024,8816,8782.0
104,san marcos,2024,455,396.0


## Appendix: City of San Diego permits (optional)

Not part of the core pipeline - doesn't feed jurisdiction_year_dataset or
any export. Used only to validate APR's San Diego permit numbers against
the City's own system (overlap check below).

## City of San Diego permits

In [136]:
permits_dir = RAW_DIR.parent / "sandiego_permits"
permits_dir.mkdir(parents=True, exist_ok=True)

permits_frames = []
for label in ["active", "closed"]:
    raw_path = permits_dir / f"{label}_approvals_raw.csv"
    if not raw_path.exists():
        print(f"Missing: {raw_path}")
        print("Download from https://data.sandiego.gov/datasets/development-permits-set2/ and save it there.")
        continue
    df = pd.read_csv(raw_path, low_memory=False)
    df["approval_status"] = label
    permits_frames.append(df)

if permits_frames:
    sd_permits_raw = pd.concat(permits_frames, ignore_index=True)
    print(sd_permits_raw.shape)
else:
    sd_permits_raw = pd.DataFrame()
sd_permits_raw.head()


(1245947, 55)


,DEVELOPMENT_ID,PROJECT_ID,PROJECT_TYPE,PROJECT_STATUS,PROJECT_PROCESSING_CODE,PROJECT_CREATE_DATE,PROJECT_DEEMEDCOMPLETE_DATE,PROJECT_TRUST_ACCOUNT_NO,PROJECT_TITLE,PROJECT_SCOPE,...,APPROVAL_ADU_TOTAL,APPROVAL_JADU_EXTREMELY_LOW,APPROVAL_JADU_VERY_LOW,APPROVAL_JADU_LOW,APPROVAL_JADU_MODERATE,APPROVAL_JADU_ABOVE_MODERATE,APPROVAL_JADU_BONUS,APPROVAL_JADU_TOTAL,APPROVAL_PERMIT_HOLDER,approval_status
0,69849.0,84160,NaN,Closed,Standard,2005-09-16,NaN,NaN,TCP 43063,GASLAMP STREET REHABILITATION PROJECT PHASE 1 ...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,active
1,66412.0,79510,NaN,Inspecting,Standard,2005-07-27,2005-07-27,NaN,Eddie Bauer T.I.Permit,5984 sq ft tenant improvement for Eddie Bauer ...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Arrow Automatic Fire Sprinkler, Arrow Automat...",active
2,69631.0,83867,NaN,Closed,Standard,2005-09-14,2005-09-14,NaN,Robinson Tenant Improvement,SOUTHEASTERN SAN DIEGO.Building Permit. Repair...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,active
3,67560.0,83802,NaN,Inspecting,Standard,2005-09-13,2005-09-13,NaN,Metro C & O - 33rd St Gate,SESDPD; - I-1(Customer provided PIC by PGB) ;R...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,active
4,41510.0,84276,NaN,In Review,Standard,2005-09-19,2005-09-23,NaN,4001 Illinois St Fourplex,GREATER NORTH PARK. Building Permit for new ...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,active


In [137]:
print(sd_permits_raw.columns.tolist())

['DEVELOPMENT_ID', 'PROJECT_ID', 'PROJECT_TYPE', 'PROJECT_STATUS', 'PROJECT_PROCESSING_CODE', 'PROJECT_CREATE_DATE', 'PROJECT_DEEMEDCOMPLETE_DATE', 'PROJECT_TRUST_ACCOUNT_NO', 'PROJECT_TITLE', 'PROJECT_SCOPE', 'JOB_ID', 'JOB_DRAWING_NUMBER', 'GIS_ADDRESS', 'GIS_APN', 'JOB_BC_CODE', 'JOB_BC_CODE_DESCRIPTION', 'GIS_LATITUDE', 'GIS_LONGITUDE', 'APPROVAL_ID', 'APPROVAL_CATEGORY_CODE', 'APPROVAL_PROCESSING_CODE', 'APPROVAL_TYPE', 'APPROVAL_STATUS', 'APPROVAL_SCOPE', 'APPROVAL_CREATE_DATE', 'APPROVAL_ISSUE_DATE', 'APPROVAL_CLOSE_DATE', 'APPROVAL_EXPIRE_DATE', 'APPROVAL_VALUATION', 'APPROVAL_DU_NET_CHANGE', 'APPROVAL_STORIES', 'APPROVAL_FLOOR_AREA', 'APPROVAL_DU_EXTREMELY_LOW', 'APPROVAL_DU_VERY_LOW', 'APPROVAL_DU_LOW', 'APPROVAL_DU_MODERATE', 'APPROVAL_DU_ABOVE_MODERATE', 'APPROVAL_DU_FUTURE_DEMO', 'APPROVAL_DU_BONUS', 'APPROVAL_ADU_EXTREMELY_LOW', 'APPROVAL_ADU_VERY_LOW', 'APPROVAL_ADU_LOW', 'APPROVAL_ADU_MODERATE', 'APPROVAL_ADU_ABOVE_MODERATE', 'APPROVAL_ADU_BONUS', 'APPROVAL_ADU_TOTAL', 

In [138]:
sd_permits_raw["APPROVAL_ISSUE_DATE"] = pd.to_datetime(sd_permits_raw["APPROVAL_ISSUE_DATE"], errors="coerce")

DU_TIER_COLS = [
    "APPROVAL_DU_EXTREMELY_LOW", "APPROVAL_DU_VERY_LOW", "APPROVAL_DU_LOW",
    "APPROVAL_DU_MODERATE", "APPROVAL_DU_ABOVE_MODERATE",
]
ADU_JADU_COLS = ["APPROVAL_ADU_TOTAL", "APPROVAL_JADU_TOTAL"]

sd_permits_raw["du_tier_total"] = sd_permits_raw[DU_TIER_COLS].fillna(0).sum(axis=1)
sd_permits_raw["adu_jadu_total"] = sd_permits_raw[ADU_JADU_COLS].fillna(0).sum(axis=1)

has_du_impact = (sd_permits_raw["du_tier_total"] != 0) | (sd_permits_raw["adu_jadu_total"] != 0)
in_target_year = sd_permits_raw["APPROVAL_ISSUE_DATE"].dt.year == TARGET_YEAR

sd_permits_housing = sd_permits_raw[has_du_impact & in_target_year].copy()
print(f"Housing-relevant permits, {TARGET_YEAR}:", len(sd_permits_housing))
print("Of", len(sd_permits_raw), "total raw rows")
sd_permits_housing[["PROJECT_TITLE", "JOB_BC_CODE_DESCRIPTION", "du_tier_total", "adu_jadu_total"]].head(10)


Housing-relevant permits, 2025: 1989
Of 1245947 total raw rows


,PROJECT_TITLE,JOB_BC_CODE_DESCRIPTION,du_tier_total,adu_jadu_total
262976,General-Express-Building Construction:6757/Roe...,Five or More Family Apt,3.0,9.0
263074,General-Standard-Building Construction:1874/Su...,Add/Alt Companion Unit/Acc Apt,0.0,1.0
263207,General-Standard-Combination Building Permit:2...,Add/Alt Companion Unit/Acc Apt,0.0,2.0
263216,Actively Managed-Express-Building Construction...,Five or More Family Apt,126.0,0.0
263424,General-Standard-Building Construction:4248/Arden,Add/Alt Companion Unit/Acc Apt,0.0,1.0
263429,Actively Managed-Express-Building Construction...,Five or More Family Apt,196.0,0.0
263480,General-Standard-Building Construction:1405/Grove,Add/Alt Companion Unit/Acc Apt,0.0,1.0
263649,Building Construction - Master Plan MDU:10202/...,NaN,5.0,0.0
263652,Building Construction - Master Plan MDU:10202/...,NaN,5.0,0.0
263720,General-Standard-Building Construction:3321/52nd,Add/Alt Companion Unit/Acc Apt,0.0,1.0


## Check APR vs. City of SD permit overlap

In [139]:
apr_sd_only = sd_apr_target_year[sd_apr_target_year["jur_clean"] == "san diego"]
apr_sd_units = apr_sd_only["bp_units_row"].sum()

city_permits_total = sd_permits_housing["du_tier_total"].sum() + sd_permits_housing["adu_jadu_total"].sum()

print(f"APR, City of San Diego, {TARGET_YEAR} permitted units: {apr_sd_units:.0f}")
print(f"City of SD permit system, {TARGET_YEAR} units (DU tiers + ADU/JADU): {city_permits_total:.0f}")
print(f"Ratio (city system / APR): {city_permits_total / apr_sd_units:.2f}" if apr_sd_units else "APR total is zero")


APR, City of San Diego, 2025 permitted units: 7842
City of SD permit system, 2025 units (DU tiers + ADU/JADU): 8177
Ratio (city system / APR): 1.04


In [140]:
apr_sd_apns = set(apr_sd_only["APN"].dropna().astype(str).str.strip())
city_apns = set(sd_permits_housing["GIS_APN"].dropna().astype(str).str.strip())

overlap_apns = apr_sd_apns & city_apns
print(f"APR SD APNs ({TARGET_YEAR}): {len(apr_sd_apns)}")
print(f"City permit APNs ({TARGET_YEAR}): {len(city_apns)}")
print(f"APNs appearing in both: {len(overlap_apns)}")
print(f"Share of City permit APNs also in APR: {len(overlap_apns) / len(city_apns):.1%}" if city_apns else "no city APNs")


APR SD APNs (2025): 1333
City permit APNs (2025): 1426
APNs appearing in both: 1325
Share of City permit APNs also in APR: 92.9%


**Finding:** APR and City of SD permits report largely the same
underlying activity (totals close, ~93% APN match) - don't sum them.
Doesn't apply to the other 17 jurisdictions; APR is their only source.

## Missing data, unclear fields, and source limitations

In [141]:
data_quality_log = pd.DataFrame([
    {"type": "Missing data", "source": "APR Table A2", "item": "bp_affordable_share / co_affordable_share",
     "note": "NaN when bp_units_total or co_units_total is 0 for that jurisdiction-year (0/0), not a data error - means zero permits/completions that year, not unknown affordability"},
    {"type": "Missing data", "source": "APR Table A2", "item": "NOTES, LATITUDE/LONGITUDE, DR_TYPE, FIN_ASSIST_NAME, PRIOR_APN",
     "note": "Conditional fields - only populated for specific project types (e.g. DR_TYPE only for deed-restricted units). Sparse by design, not incomplete"},
    {"type": "Missing data", "source": "City of SD permits", "item": "ADU/JADU and income-tier DU columns",
     "note": "Blank on non-residential permit rows (electrical, plumbing, signage, etc.) - expected, filtered out of sd_permits_housing"},
    {"type": "Missing data", "source": "APR Table F (preservation)", "item": "entire table",
     "note": "Not yet loaded into this workstream. Exists as a processed file in the dashboard prototype repo (sd_apr_f_preservation_city_year.csv) but not pulled live here"},
    {"type": "Missing data", "source": "This workstream", "item": "NOAH (naturally occurring affordable housing) loss estimate",
     "note": "No published dataset exists for this - would need to be derived from ACS rent + building-age data. Not started"},
    {"type": "Missing data", "source": "HCD APR (general)", "item": "current reporting year, some jurisdictions",
     "note": "Jurisdictions can file late; a given year's data may be incomplete for months after the April 1 deadline. Check the 'not found' warning printed when loading before trusting a fresh pull"},
    {"type": "Missing data", "source": "HCD APR", "item": "units under construction",
     "note": "Not tracked anywhere in HCD's APR data - no table captures this milestone. Application, entitlement, permit, and completion stages are all available (Table A and Table A2), but under-construction status is not"},
    {"type": "Missing data", "source": "APR Table A2, City of San Diego", "item": "ent_units_total near-zero relative to bp_units_total",
     "note": "San Diego reported only 3 entitled units in 2025 against 7,842 permitted units - an unusually large gap for the county's largest jurisdiction. Likely a self-reporting gap in San Diego's own APR submission for the entitlement stage specifically, not evidence that entitlement activity actually stopped"},
    {"type": "Unclear field", "source": "APR Table A2", "item": "*_DR / *_NDR column suffixes",
     "note": "Deed Restricted / Non-Deed Restricted (whether the affordability requirement is legally recorded on the property). Not explained in the CSV itself - confirmed from HCD's separate Table A2 data dictionary (.docx)"},
    {"type": "Unclear field", "source": "APR Table A2 vs. RHNA6", "item": "VLOW vs. VLI",
     "note": "Same income tier (Very Low Income), different abbreviation between two HCD datasets. Easy to miss if joining/comparing by tier name"},
    {"type": "Unclear field", "source": "RHNA6 progress report", "item": "what \"units reported\" actually measures",
     "note": "RHNA progress in this dataset is based on BUILDING PERMITS ISSUED, not completed units - confirmed via HCD's own APR guidance (\"only building permits are used for the purposes of determining progress towards RHNA\"). Entitlements and completions are tracked elsewhere in the APR but do not count toward RHNA credit"},
    {"type": "Unclear field", "source": "City of SD permits", "item": "APPROVAL_DU_NET_CHANGE",
     "note": "Misleadingly named - sums to exactly 0 across a full year of data, unreliable. Real unit counts live in the separate income-tier APPROVAL_DU_* columns instead"},
    {"type": "Resolved / corrected", "source": "APR Table A2", "item": "NO_ENTITLEMENTS / NO_BUILDING_PERMITS / NO_OTHER_FORMS_OF_READINESS",
     "note": "CORRECTED: previously mischaracterized as flags. Per HCD's official APR instructions, these mean \"Number Of\" - auto-populated total-unit-count fields calculated by HCD from the same per-tier income columns this notebook sums independently. Cross-check against ent_units_total / bp_units_total / co_units_total now matches exactly for all 19/19 jurisdictions, all three stages"},
    {"type": "Resolved / corrected", "source": "This notebook", "item": "ENT_INCOME_COLS incorrectly included EXTR_LOW_INCOME_UNITS",
     "note": "A loose text search (\"contains INCOME, not BP_/CO_ prefixed\") swept in an unrelated Table A2 field, inflating ent_units_total beyond what the 4 tier columns summed to. Caused an apparent 55-unit Escondido anomaly that was never a real data-source issue - fixed by building the column list explicitly from the known tier structure instead of a text search"},
    {"type": "Missing data", "source": "APR Table A", "item": "application_units_total vs. TOT_PROPOSED_UNITS small discrepancies",
     "note": "14 of 19 jurisdictions match exactly; 5 show small gaps (largest: Escondido, 88 units / ~16% relative). Likely reflects units reported in TOT_PROPOSED_UNITS without a corresponding income-tier breakdown yet, rather than a calculation error"},
    {"type": "Missing data", "source": "APR Table A (historical, 2018-2020)", "item": "one jurisdiction per year with zero applications reported",
     "note": "Lemon Grove (2018), Imperial Beach (2019), and San Marcos (2020) each show zero application rows for that year - filled with 0 in the Power BI export via fill_missing_jurisdiction_years(), but genuinely absent from HCD's raw Table A data for those specific jurisdiction-years"},
    {"type": "Unclear field", "source": "HCD / DOF / dashboard prototype", "item": "County jurisdiction naming",
     "note": "Appears as 'Unincorporated' (DOF), 'SAN DIEGO COUNTY' (APR/RHNA), and 'County of San Diego' in various places - required manual normalization to a single consistent key ('unincorporated san diego county') across this notebook"},
    {"type": "Source limitation", "source": "APR / RHNA (HCD)", "item": "self-reported",
     "note": "Not independently verified by HCD. Quality, completeness, and filing timeliness vary by jurisdiction"},
    {"type": "Source limitation", "source": "RHNA6 progress report", "item": "cumulative only",
     "note": "No year-by-year breakdown - reports cycle-to-date totals only (2021-2029), can't see year-over-year pace toward the target from this file alone"},
    {"type": "Source limitation", "source": "City of SD permits", "item": "single-jurisdiction coverage",
     "note": "Only covers the City of San Diego, not the other 17 jurisdictions. Also overlaps with APR for San Diego specifically (~1.04 ratio, 93% APN match) - don't sum the two for San Diego totals"},
    {"type": "Source limitation", "source": "DOF E-5", "item": "annual point-in-time estimate",
     "note": "January 1 snapshot, not real-time. Uses different methodology than ACS, so the two won't match exactly even for the same year"},
    {"type": "Source limitation", "source": "Census ACS", "item": "one year behind target year",
     "note": "Newest available vintage is 2020-2024 (\"2024\" data) while APR/DOF/permits target 2025 - ACS structurally cannot produce 2025 data until ~Dec 2026/Jan 2027. Also a 5-year rolling estimate with margins of error, largest for small jurisdictions like Del Mar (~3,900 population)"},
    {"type": "Source limitation", "source": "Dashboard prototype (housing-dashboard-prototype repo)", "item": "stale baseline for validation",
     "note": "The prototype's sd_apr_a2_city_year_supply.csv only covers 2018-2024 - it has no 2025 data yet, so this notebook's 2025 pull currently has nothing to validate against for that year. Not an error in this notebook; the prototype simply hasn't been refreshed with 2025 APR data"},
    {"type": "Source limitation", "source": "HCD APR (general)", "item": "historical years get amended after publication",
     "note": "Validating 2018-2024 against the dashboard prototype found 9 of 126 city-year rows differ (out of 18 cities x 7 years). In every case this notebook's fresh pull is HIGHER than the prototype's older snapshot, never lower, and gaps are small (3-59 units) - consistent with HCD amending past years' data (late filings, corrections) after initial publication. Largest gap: San Marcos 2024 (455 vs. 396, ~13%)"},
])

data_quality_log.to_csv(DOCS_DIR / "data_quality_limitations.csv", index=False)
data_quality_log


,type,source,item,note
0,Missing data,APR Table A2,bp_affordable_share / co_affordable_share,NaN when bp_units_total or co_units_total is 0...
1,Missing data,APR Table A2,"NOTES, LATITUDE/LONGITUDE, DR_TYPE, FIN_ASSIST...",Conditional fields - only populated for specif...
2,Missing data,City of SD permits,ADU/JADU and income-tier DU columns,Blank on non-residential permit rows (electric...
3,Missing data,APR Table F (preservation),entire table,Not yet loaded into this workstream. Exists as...
4,Missing data,This workstream,NOAH (naturally occurring affordable housing) ...,No published dataset exists for this - would n...
5,Missing data,HCD APR (general),"current reporting year, some jurisdictions",Jurisdictions can file late; a given year's da...
6,Missing data,HCD APR,units under construction,Not tracked anywhere in HCD's APR data - no ta...
7,Missing data,"APR Table A2, City of San Diego",ent_units_total near-zero relative to bp_units...,San Diego reported only 3 entitled units in 20...
8,Unclear field,APR Table A2,*_DR / *_NDR column suffixes,Deed Restricted / Non-Deed Restricted (whether...
9,Unclear field,APR Table A2 vs. RHNA6,VLOW vs. VLI,"Same income tier (Very Low Income), different ..."
